<a href="https://colab.research.google.com/gist/hirokazu/f94c539ba208784c9ec9cfc18acffd73/minimax_h3_on_colab_t4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax H3 (video + synchronized audio) on a stock Colab T4

This notebook runs **MiniMax H3** -- a 19.5 GB audio-video diffusion transformer --
end to end on Colab's cheapest GPU runtime (**Tesla T4**, sm75, **12.7 GB host RAM**,
2 vCPU) and produces a 5.2-second MP4 **with synchronized audio**.

| config | resolution | wall time (T4) |
|---|---|---|
| 6 steps (recommended) | 864x480 | ~35 min |
| 4 steps | 960x544 | ~40 min |

## Why this is supposed to be impossible

The weights total **39.6 GB** (DiT 19.5 + text encoder 14.6 + VAEs 5.4) against
12.7 GB of host RAM. The binding constraint on Colab is **host RAM, not VRAM** --
the T4 actually has more VRAM (14.9 GB) than the box has RAM, which is backwards
compared to a desktop, so the usual VRAM-centric H3 optimizations don't help here.

The fix is a **three-stage split**: encode, sample, and decode each run in a
*separate process*, handing off through tiny files (a ~5 MB conditioning blob and
a ~5 MB latent). Peak RSS becomes max(stage) = **9.2 GB** instead of
sum(stages) = 21.9 GB.

```
encode   text encoder 14.6 GB (mmap-streamed)  -> conditioning blob (~5 MB)
sample   DiT GGUF Q4_K_M 10.8 GB + Turbo LoRA  -> latent (~5 MB)
decode   video VAE 4.9 GB + audio VAE 0.6 GB   -> MP4 with audio
```

## What's inside (the parts that took blood to learn)

- **bf16, not fp16.** fp16 overflows *mathematically*: per-module tracing showed
  `blocks.0` residual writes hitting absmax ~35k against fp16's 65504 cap, with no
  normalization downstream to save it. Not a kernel bug -- the values just don't fit.
- **The GGUF must declare architecture `wan`.** ComfyUI-GGUF validates
  `general.architecture` against a hardcoded list; quants that omit the field are
  rejected before a single tensor is read. The Abiray pruned quants pass.
- **Turbo LoRA at strength 1.0.** The file ships no alpha tensor (alpha = rank);
  the 0.094 figure circulating online belongs to a different file. On a pruned base
  you need the author's node pack -- a plain LoRA loader silently half-applies it.
  Its dual flow schedules (video/audio) are why the audio stays in sync at 4-6 steps.
- **A cliff guard** projects the finish time from live sampler progress and cancels
  runs that cannot fit the time budget -- so a bad config costs 13 minutes, not 10 hours.
- **A per-step perf probe and a same-session benchmark harness** (in the full
  research notebooks) busted our own myth: what looked like a performance cliff
  between the 864x480 and 960x544 gears was **VM-to-VM variance (+-20%)** --
  measured in one session, scaling is exactly quadratic attention + linear MLP
  (within 2%). Benchmark on one VM, always.

Every downloaded weight is SHA-256 verified. Every pin (ComfyUI, ComfyUI-GGUF,
Turbo nodes) is a fixed commit. The sampling graph is built from the live server's
schema, not copied from a workflow JSON.

## How to run

1. Runtime -> **T4 GPU**.
2. In Stage 0, tick the **license checkboxes** (this model is license-gated; read
   the linked terms) and `RUN_MODEL_DOWNLOAD`.
3. **Run all.** First run downloads ~17.6 GB of weights (DiT + LoRA + VAEs), plus
   a **one-time ~15 GB text encoder** if no conditioning blob exists yet for your
   prompt (it is cached to your Drive afterwards, so this happens once per prompt).
4. The MP4 lands in `MyDrive/H3/MiniMax-H3-Native-Colab/video/`.

Change `PROMPT` to generate something else -- each new prompt re-encodes once.

## Credits

[ComfyUI](https://github.com/Comfy-Org/ComfyUI) *
[ComfyUI-GGUF](https://github.com/city96/ComfyUI-GGUF) *
[Abiray/MiniMax-H3-Pruned-GGUF](https://huggingface.co/Abiray/MiniMax-H3-Pruned-GGUF) *
[larryvrh/MiniMax-H3-Turbo-Lora](https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora) +
[node pack](https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo) *
[Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3)

Weights are downloaded under the MiniMax H3 license you accept in Stage 0.


## Stage 0 -- Configuration, environment, conditioning-blob lookup

Everything is decided here (gear, dtype, DiT, LoRA, decode). No patch cells.


In [ ]:
# =============================================================================
# 0. Verificação Inicial de GPU (Google Colab)
# =============================================================================
import torch
import sys

print('🔍 Verificando acelerador de hardware...')
if not torch.cuda.is_available():
    raise RuntimeError(
        '\n❌ ERRO: Nenhuma GPU detectada!\n\n'
        'Para configurar corretamente:\n'
        '1. No menu superior do Google Colab, clique em "Ambiente de execução" -> "Alterar tipo de ambiente de execução"\n'
        '2. Selecione "GPU T4" como Acelerador de Hardware\n'
        '3. Clique em "Salvar" e execute novamente as células.'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gib = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f'✅ GPU pronta para uso: {gpu_name} ({vram_gib} GiB VRAM)')


In [ ]:
# Stage 0 - configuration, environment, and finding the conditioning blob.
import hashlib
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path

import psutil

# --- License gate, unchanged from the original notebook ---------------------
#   https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE
#   https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/QA-about-License.md
I_HAVE_READ_AND_ACCEPTED_MINIMAX_H3_LICENSE = True  # @param {type:"boolean"}
MINIMAX_LICENSE_BASIS = "applicable_territory"  # @param ["", "applicable_territory", "separate_minimax_authorization"]
I_CONFIRM_RUNTIME_AND_USE_LOCATIONS_ARE_COVERED = True  # @param {type:"boolean"}
I_CONFIRM_ACCEPTABLE_USE_AND_OUTPUT_RULES = True  # @param {type:"boolean"}
I_ACCEPT_RESPONSIBILITY_FOR_OUTPUTS = True  # @param {type:"boolean"}
RUN_MODEL_DOWNLOAD = True  # @param {type:"boolean"}

# Which quantised DiT to sample with. The encode stage peaked at 9.17 GiB against
# 11.18 GiB available, so much past 9 GiB of weights is a gamble -- and unlike the
# encoder, the DiT is read on every one of the 20 steps, so it has to fit rather
# than stream.
# Only the abiray_* entries load: see the gguf_arch note on DIT_CHOICES below.
DIT_CHOICE = "abiray_Q4_K_M"  # @param ["abiray_Q4_K_M", "abiray_Q3_K_M", "unsloth_UD_Q3_K_XL", "unsloth_Q3_K", "unsloth_UD_Q2_K_XL", "unsloth_Q2_K"]
STEPS = 6  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
SAMPLER = "res_multistep"  # @param {type:"string"}
SCHEDULER = "simple"  # @param {type:"string"}

# The conditioning blob is 5 MiB and costs a 14.61 GiB download plus an encode to
# regenerate, which makes ephemeral disk the wrong place for it. Leave
# CONDITIONING_BLOB empty to search: Drive first, then this runtime's own stores.
CONDITIONING_BLOB = ""  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}
REGENERATE_CONDITIONING_IF_MISSING = True  # @param {type:"boolean"}

CLIFF_PROBE_WINDOW_S = 120  # @param {type:"integer"}
CLIFF_PROBE_MAX_WINDOWS = 2  # @param {type:"integer"}
CLIFF_PROJECTED_BUDGET_S = 3600  # @param {type:"integer"}
SAMPLER_START_TIMEOUT_S = 1800  # @param {type:"integer"}
# Decoding holds no DiT and the latent is small, but the video VAE is 4.85 GiB
# and this budget also covers restarting ComfyUI and loading it.
DECODE_TIMEOUT_S = 1800  # @param {type:"integer"}
# Inspect every Nth decoded frame for brightness and texture. A finite latent can
# still decode to black, so "the file exists" is checked separately from
# "the video has a picture in it".
VIDEO_INSPECT_EVERY = 8  # @param {type:"integer"}

# Compute dtype for the DiT. Measured 2026-08-11 on this runtime at 736x416/124f:
#   fp16 -> 36.6 s/step but the latents come out NaN. H3's DiT is bf16-trained and
#           fp16's exponent range (max 65504) is too small; the log warns
#           "Mismatch dtype ... input dtype = c10::Half" in rms_norm.
#   bf16 -> 207-214 s/step and finite. T4 has no native bf16, so this is emulated,
#           which costs about 5.8x.
# Earlier versions set this with a patch cell, and three runs in a row were spoiled
# by the patched state disagreeing with the config. It is a Stage 0 field now, and
# Stage 3 asserts that what ComfyUI actually got matches what was asked for.
UNET_DTYPE = "bf16"  # @param ["bf16", "fp16", "fp32", "default"]

# Compute RMSNorm in fp32 while everything else stays on fp16 tensor cores.
#
# rms_norm is x * rsqrt(mean(x^2) + eps) * weight, and in fp16 x^2 overflows once
# |x| > 256 (256^2 = 65536 > 65504) -- a level video DiT activations reach easily.
# That is the most likely source of the NaN seen in fp16 at both Q3_K_M and Q4_K_M.
#
# Reachable: norm1 / norm2 / final_norm, which go through ComfyUI's RMSNorm.
#   ComfyUI-GGUF never overrides RMSNorm, so GGMLOps inherits it unchanged.
# NOT reachable: Attention's q_norm / k_norm, which on the rope path call
#   comfy_kitchen's fused ck.rms_rope_split_half_() and bypass the module. If the
#   overflow lives there, this changes nothing.
#
# The patch only acts when the input is fp16; bf16 already has fp32's exponent
# range, so leaving this on costs nothing there.
RMSNORM_FP32 = True  # @param {type:"boolean"}

# Find WHERE the value goes non-finite instead of guessing which op to upcast.
# Hooks every submodule of the DiT and records the first one whose output is
# non-finite while its input is still finite -- that module creates the value.
# One GPU sync per module until the first bad one, so run it with STEPS = 1.
# Leave it off for a real run.
NAN_TRACE = False  # @param {type:"boolean"}

# Cliff diagnosis (the step between gear 0.4 and 0.5). Times attention/MLP per
# step with CUDA events, censuses every SDPA call (shapes, dtype, dispatcher's
# kernel choice), and watches the allocator counters -- the three candidate
# mechanisms leave three different fingerprints. One synchronize per model
# forward: milliseconds against a minutes-long step, so probed timings remain
# measurements and the guard needs no widening. Run STEPS = 2 at gear 0.4 and
# again at 0.5, then compare the H3_PERF_STEP lines (step 1 includes warmup;
# read step 2).
PERF_PROBE = False  # @param {type:"boolean"}

# Measured 2026-08-11: at fp16 the FIRST module to produce a tensor, condition_proj
# (a Linear at the entrance of the DiT), overflows. Its input is already finite at
# absmax 15424 and the output goes inf. Everything else was downstream of that.
#
# ComfyUI's own H3 code builds video_patch_proj and audio_patch_proj with an
# explicit dtype=torch.float32 but leaves condition_proj at the model dtype --
# harmless in bf16, fatal in fp16.
#
# Upcasting condition_proj's INPUT is enough: comfy's manual cast and GGUF's
# dequant both follow the input dtype, so the whole text branch (condition_proj ->
# token_refiner) runs fp32 and comes back through final_norm, which is a RMSNorm --
# so the handoff to the fp16 sequence buffer is safe by construction, not by luck.
# The text branch is ~512 tokens against 37*26*46 video tokens, so it is nearly free.
FP32_TEXT_BRANCH = True  # @param {type:"boolean"}

# The Turbo LoRA turns ~20 steps into 4-8. Its README and its own file metadata
# agree that there is NO alpha tensor and the update is a plain W + B@A, i.e.
# alpha = rank, so strength 1.0 is the tuned value -- the 0.094 figure that
# circulates belongs to a file with alpha/rank = 1/8 and does not apply here.
#
# It needs the author's node pack, not LoraLoaderModelOnly: on a pruned base the
# adaln update cannot be a weight patch (the time-embedding space is collapsed
# into a curve), so the node re-injects it at run time. Our DiT is FL2VA-pruned.
#
# 4 steps is the documented MINIMUM; 4-8 is the useful range and 6-8 looks
# noticeably better. Past 8 it stops helping. Scheduler stays "simple".
TURBO_LORA = "v4_step600_ema"  # @param ["v4_step600_ema", "v1_850_ema", "off"]
LORA_STRENGTH = 1.0  # @param {type:"number"}
# low_vram merges the LoRA into the weights instead of applying it at run time.
# Lowest peak VRAM, "a bit softer on quantized bases" -- and ours is Q4_K.
LORA_LOW_VRAM = False  # @param {type:"boolean"}

# Decode the sampled latent into an actual MP4. This restarts ComfyUI first: the
# DiT is 10.77 GiB and the video VAE is 4.85 GiB, and the whole point of the
# three-stage split is that peak is max(stage) rather than sum(stages).
DECODE_TO_MP4 = True  # @param {type:"boolean"}

# A traced step is slower than a real one -- one GPU sync per module until the
# first non-finite. The cliff guard would read that as a cliff and cancel before
# the trace exists, so widen the window when tracing. Timings from a traced run
# mean nothing anyway; the trace is the whole output.
if NAN_TRACE:
    CLIFF_PROBE_WINDOW_S = max(CLIFF_PROBE_WINDOW_S, 1800)
    CLIFF_PROJECTED_BUDGET_S = max(CLIFF_PROJECTED_BUDGET_S, 3600)
    print(f"NAN_TRACE widened the guard: window {CLIFF_PROBE_WINDOW_S}s, "
          f"budget {CLIFF_PROJECTED_BUDGET_S}s (traced timings are not measurements)")
ENABLE_TRITON_BACKEND = True  # @param {type:"boolean"}

UNET_DTYPE_FLAGS = {"bf16": "--bf16-unet", "fp16": "--force-fp16",
                    "fp32": "--fp32-unet", "default": None}
if UNET_DTYPE not in UNET_DTYPE_FLAGS:
    raise RuntimeError(f"Unknown UNET_DTYPE {UNET_DTYPE!r}; pick one of {sorted(UNET_DTYPE_FLAGS)}")
UNET_DTYPE_FLAG = UNET_DTYPE_FLAGS[UNET_DTYPE]

COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI.git"
COMFY_COMMIT = "9a9fdb10ed144ce760d9682cb247526ea23cc525"
# ComfyUI-GGUF rejects any architecture not in its hardcoded IMG_ARCH_LIST, and
# minimax_h3 is not in it. Stage 1 adds it with a guarded patch. Pinned so the
# anchor is known to appear exactly once.
GGUF_REPO = "https://github.com/city96/ComfyUI-GGUF.git"
GGUF_COMMIT = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"
GGUF_LOADER_ANCHOR = '"qwen_image"}'
GGUF_LOADER_PATCHED = '"qwen_image", "minimax_h3"}'

MODEL_REPO = "Comfy-Org/MiniMax-H3"
MODEL_REVISION = "0543966fbdce5ba05709a8f2031c94bdba629b4a"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# Needed only when the blob has to be regenerated.
TEXT_ENCODER_FILE = "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE_FILE = "vae/minimax_h3_video_vae_fp16.safetensors"
ENCODE_FILES = {
    TEXT_ENCODER_FILE: {
        "bytes": 15687142551,
        "sha256": "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6"},
    VIDEO_VAE_FILE: {
        "bytes": 5207808496,
        "sha256": "7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522"},
}

# All FL2VA, all pruned. sha256 is the HF LFS oid.
#
# gguf_arch is what the file declares in general.architecture. ComfyUI-GGUF checks
# that against a hardcoded IMG_ARCH_LIST, and a file declaring nothing falls into a
# compatibility branch that guesses from tensor names and fails for H3:
#   ValueError: This model is not currently supported - (Unknown model architecture!)
# The repos that work declare "wan" -- which is wrong as a description but already
# in the list, and ComfyUI identifies the real architecture from the state dict
# anyway ("Requested to load MiniMaxH3"). Verified 2026-08-11.
DIT_CHOICES = {
    "unsloth_Q2_K": {
        "gguf_arch": None,
        "repo": "unsloth/MiniMax-H3-GGUF", "file": "minimax_h3_fl2va_pruned-Q2_K.gguf",
        "bytes": 6724190304,
        "sha256": "696866a3c98ea0bb7e96b8a273aafb7c588ca8e6fb669303956eb270cedb68aa"},
    "unsloth_UD_Q2_K_XL": {
        "gguf_arch": None,
        "repo": "unsloth/MiniMax-H3-GGUF", "file": "minimax_h3_fl2va_pruned-UD-Q2_K_XL.gguf",
        "bytes": 8063029344,
        "sha256": "cfe0795c00ab6e6ebf8c64fe4574f45a828e8a93e0876bca704e055662a9d7b8"},
    "unsloth_Q3_K": {
        "gguf_arch": None,
        "repo": "unsloth/MiniMax-H3-GGUF", "file": "minimax_h3_fl2va_pruned-Q3_K.gguf",
        "bytes": 8759328864,
        "sha256": "2c3cadbcc87ce2ed550b9be1e3b099198ba31a33a83f051411254c5149f6f59f"},
    "abiray_Q3_K_M": {
        "gguf_arch": 'wan',
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF", "file": "MiniMax-H3-FL2VA-Pruned-Q3_K_M.gguf",
        "bytes": 8902845536,
        "sha256": "c844d70dbb4df67b7d9605c669bff35d9c631bd0507143cf0276f984de32247d"},
    "unsloth_UD_Q3_K_XL": {
        "gguf_arch": None,
        "repo": "unsloth/MiniMax-H3-GGUF", "file": "minimax_h3_fl2va_pruned-UD-Q3_K_XL.gguf",
        "bytes": 9558675552,
        "sha256": "aee15137d31131b35448fad820ff757b6c137df44fdc0758b4a4636792d2b913"},
    "abiray_Q4_K_M": {
        "gguf_arch": 'wan',
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF", "file": "MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf",
        "bytes": 11564180576,
        "sha256": "d74e644906fe95b8c7bf4837ba9d3f9392925f7301675a0993d419b62a4824a3"},
}
if DIT_CHOICE not in DIT_CHOICES:
    raise RuntimeError(f"Unknown DIT_CHOICE {DIT_CHOICE!r}; pick one of {sorted(DIT_CHOICES)}")
DIT = DIT_CHOICES[DIT_CHOICE]
if DIT.get("gguf_arch") is None:
    raise RuntimeError(
        f"{DIT_CHOICE} declares no general.architecture in its GGUF header. "
        "ComfyUI-GGUF cannot load it: it falls into the compatibility branch and "
        "fails with 'Unknown model architecture!'. Measured 2026-08-11 with "
        "unsloth/MiniMax-H3-GGUF. Choose one of: "
        f"{sorted(k for k, v in DIT_CHOICES.items() if v.get('gguf_arch'))}")

# Sizes and SHA-256 read from the Hugging Face CDN headers on 2026-08-11
# (x-linked-size / x-linked-etag), not copied from a model card.
TURBO_NODE_REPO = "https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git"
TURBO_NODE_COMMIT = "55fee864dd7b2976b1c4ce3c3d5f7968f181409f"
LORA_REPO = "larryvrh/MiniMax-H3-Turbo-Lora"
TURBO_LORA_CHOICES = {
    "v4_step600_ema": {
        "repo": LORA_REPO, "file": "minimax_h3_turbo_v4_step600_ema.safetensors",
        "bytes": 779849816,
        "sha256": "5f3a626cd72c93a8b9318d6760c510bc5092d2ab13aaba1f932c5bab07a416d3",
        "note": "current best; the author's recommendation"},
    "v1_850_ema": {
        "repo": LORA_REPO, "file": "minimax_h3_turbo_4step_ema_ckpt850.safetensors",
        "bytes": 779849816,
        "sha256": "5a6eeba171cf183020a4ad48774bb2968f29f8168afd6ec17a04987f3528b4ea",
        "note": "friendlier at 4 steps with heavy motion; over-sharpens otherwise"},
}
if TURBO_LORA not in TURBO_LORA_CHOICES and TURBO_LORA != "off":
    raise RuntimeError(f"Unknown TURBO_LORA {TURBO_LORA!r}; pick one of "
                       f"{sorted(TURBO_LORA_CHOICES) + ['off']}")
LORA = TURBO_LORA_CHOICES.get(TURBO_LORA)

VAE_FILES = {
    "video": {"file": "vae/minimax_h3_video_vae_fp16.safetensors",
              "bytes": 5207808496,
              "sha256": "7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522"},
    "audio": {"file": "vae/minimax_h3_audio_vae_fp32.safetensors",
              "bytes": 605254808,
              "sha256": "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48"},
}

# The prompt is part of the conditioning key: changing it makes Stage 2 encode a
# new conditioning blob (one-time ~15 GB text-encoder download, then cached).
PROMPT = "A locked-off medium shot of a red rubber ball bouncing three times on a wooden floor in a quiet empty room. Each impact produces a clear synchronized rubber thump. Natural daylight, simple background, no camera movement, no dialogue."  # @param {type:"string"}
FPS = 24

# The resolution gears from the porting spec. Both dimensions divisible by 32.
# Measuring where the cliff is means sampling the same clip at several of these,
# so the geometry is chosen by name rather than typed in as three numbers -- a
# mistyped width silently becomes a different conditioning key and a 19.46 GiB
# re-encode.
GEAR_TABLE = {
    "0.2": (608, 352),
    "0.3": (736, 416),
    "0.4": (864, 480),
    "0.5": (960, 544),
    "1.0": (1344, 768),
}
GEAR = "0.4"  # @param ["0.2", "0.3", "0.4", "0.5", "1.0"]
# Campaign override: Stage 7 re-executes this cell with GEAR_OVERRIDE set to
# walk several gears in one session. A manual run leaves it unset, so the form
# value above wins. Stage 7 deletes the override when the campaign ends.
if "GEAR_OVERRIDE" in globals():
    GEAR = str(globals()["GEAR_OVERRIDE"])
    print(f"GEAR_OVERRIDE active: GEAR = {GEAR} (campaign run)")
CLIP_SECONDS = 5.0  # @param {type:"number"}


def frame_count(seconds):
    """Frames for a duration, rounded up to the next legal count.

    The model silently rounds a count that does not satisfy n % 17 == 5 -- no
    error, no warning -- so the geometry that comes back would not be the one
    that was asked for. This is the formula the official workflow uses.
    """
    n = max(5, round(seconds * FPS))
    return n + (5 - (n % 17)) % 17


if GEAR not in GEAR_TABLE:
    raise RuntimeError(f"Unknown GEAR {GEAR!r}; pick one of {sorted(GEAR_TABLE)}")
GEOMETRY = {"width": GEAR_TABLE[GEAR][0], "height": GEAR_TABLE[GEAR][1],
            "length": frame_count(CLIP_SECONDS)}
# Relative cost against gear 0.3, the one that is measured. Attention is
# quadratic in tokens, so this is a lower bound on the time ratio, not a
# prediction -- the point of measuring the cliff is that it is not smooth.
GEAR_PIXEL_RATIO = round(
    (GEOMETRY["width"] * GEOMETRY["height"]) / (736 * 416), 3)

CONTENT = Path("/content")
WORK = CONTENT / "h3_sample_probe"
COMFY_DIR = CONTENT / "ComfyUI"
GGUF_NODE_DIR = COMFY_DIR / "custom_nodes" / "ComfyUI-GGUF"
CONTROL_NODE_DIR = COMFY_DIR / "custom_nodes" / "H3BenchmarkControl"
TURBO_NODE_DIR = COMFY_DIR / "custom_nodes" / "ComfyUI-MiniMax-H3-Turbo"
OUTPUT_DIR = COMFY_DIR / "output"
COMFY_LOG = WORK / "comfyui.log"
RESULT_PATH = WORK / "sample_probe_results.json"
CONDITIONING_STORE = WORK / "conditioning"
LATENT_STORE = WORK / "latents"

# --- Montagem e Integração Total com o Google Drive ---
DRIVE_MOUNT = CONTENT / "drive"
DRIVE_MOUNTED = (DRIVE_MOUNT / "MyDrive").is_dir()
if MOUNT_DRIVE and not DRIVE_MOUNTED:
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
        DRIVE_MOUNTED = (DRIVE_MOUNT / "MyDrive").is_dir()
    except Exception as exc:
        print(f"Drive mount failed ({type(exc).__name__}); continuing without it.")

DRIVE_H3 = DRIVE_MOUNT / "MyDrive" / "H3" / "MiniMax-H3-Native-Colab"
DRIVE_CONDITIONING_STORE = DRIVE_H3 / "conditioning"
DRIVE_VIDEO_STORE = DRIVE_H3 / "video"
CLIFF_LEDGER = DRIVE_H3 / "cliff_ledger.jsonl"

# Detecta a pasta ComfyUI no Google Drive do usuário (ComfyUI, ComfyUi, comfyui ou H3)
drive_candidates = [
    DRIVE_MOUNT / "MyDrive" / "ComfyUI",
    DRIVE_MOUNT / "MyDrive" / "ComfyUi",
    DRIVE_MOUNT / "MyDrive" / "comfyui",
    DRIVE_MOUNT / "MyDrive" / "H3",
]
DRIVE_COMFY_DIR = next((p for p in drive_candidates if p.is_dir()), DRIVE_MOUNT / "MyDrive" / "ComfyUI")
DRIVE_COMFY_MODELS = DRIVE_COMFY_DIR / "models"

if DRIVE_MOUNTED:
    MODEL_STORE = DRIVE_COMFY_MODELS
    for sub in ("diffusion_models", "unet", "loras", "vae", "text_encoders", "clip"):
        (MODEL_STORE / sub).mkdir(parents=True, exist_ok=True)
else:
    MODEL_STORE = WORK / "models"

for directory in (WORK, MODEL_STORE, CONDITIONING_STORE, LATENT_STORE, DRIVE_CONDITIONING_STORE, DRIVE_VIDEO_STORE):
    try:
        directory.mkdir(parents=True, exist_ok=True)
    except Exception:
        pass


def find_model_file(filename, subfolders=None, min_bytes=0):
    """
    Busca inteligente e universal de modelos no Drive e localmente.
    Evita baixar novamente se o modelo já existir em qualquer pasta do Drive.
    """
    if subfolders is None:
        subfolders = []
    
    candidates = []
    
    # 1. Pastas locais do ComfyUI
    candidates.append(COMFY_DIR / "models" / filename)
    for sub in subfolders:
        candidates.append(COMFY_DIR / "models" / sub / filename)
        
    # 2. Pastas em MODEL_STORE
    candidates.append(MODEL_STORE / filename)
    for sub in subfolders:
        candidates.append(MODEL_STORE / sub / filename)
    
    # 3. Pastas diretas do Google Drive
    if DRIVE_MOUNTED:
        drive_bases = [
            DRIVE_COMFY_MODELS,
            DRIVE_MOUNT / "MyDrive" / "ComfyUI" / "models",
            DRIVE_MOUNT / "MyDrive" / "ComfyUi" / "models",
            DRIVE_MOUNT / "MyDrive" / "comfyui" / "models",
            DRIVE_MOUNT / "MyDrive" / "H3" / "models",
            DRIVE_MOUNT / "MyDrive" / "models",
        ]
        for db in drive_bases:
            candidates.append(db / filename)
            for sub in subfolders:
                candidates.append(db / sub / filename)
                
    # Verifica candidatos diretos
    for p in candidates:
        try:
            if p.is_file():
                sz = p.stat().st_size
                if min_bytes == 0 or sz >= min_bytes * 0.98:
                    return p
        except Exception:
            pass
            
    # 4. Varredura no Drive caso o arquivo esteja em outra subpasta
    if DRIVE_MOUNTED:
        search_roots = [
            DRIVE_COMFY_DIR,
            DRIVE_H3,
            DRIVE_MOUNT / "MyDrive" / "ComfyUI",
            DRIVE_MOUNT / "MyDrive" / "ComfyUi",
        ]
        for root_dir in search_roots:
            try:
                if root_dir.is_dir():
                    for match in root_dir.glob(f"**/{filename}"):
                        if match.is_file() and (min_bytes == 0 or match.stat().st_size >= min_bytes * 0.98):
                            return match
            except Exception:
                pass
                
    return None

GIB = 1024 ** 3
BASE_URL = "http://127.0.0.1:8188"
RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())

if RESULT_PATH.exists():
    backup = RESULT_PATH.with_suffix(f".json.bak-{time.strftime('%Y%m%dT%H%M%S')}")
    shutil.copy2(RESULT_PATH, backup)
    try:
        PROBE = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
        print(f"Resumed earlier results ({sorted(PROBE.get('stages', {}))}); backup {backup.name}")
    except json.JSONDecodeError:
        PROBE = {"schema_version": 2, "stages": {}}
else:
    PROBE = {"schema_version": 2, "stages": {}}
PROBE.setdefault("stages", {})


def save():
    RESULT_PATH.write_text(json.dumps(PROBE, indent=2, ensure_ascii=False, default=str),
                           encoding="utf-8")


def utc_now():
    import datetime as dt
    return dt.datetime.now(dt.timezone.utc).isoformat()


def license_gate_open():
    return all((
        I_HAVE_READ_AND_ACCEPTED_MINIMAX_H3_LICENSE,
        MINIMAX_LICENSE_BASIS in {"applicable_territory", "separate_minimax_authorization"},
        I_CONFIRM_RUNTIME_AND_USE_LOCATIONS_ARE_COVERED,
        I_CONFIRM_ACCEPTABLE_USE_AND_OUTPUT_RULES,
        I_ACCEPT_RESPONSIBILITY_FOR_OUTPUTS,
    ))


CHILD_ENV_ALLOWLIST = {
    "PATH", "HOME", "LANG", "TMPDIR", "CUDA_HOME", "CUDA_PATH", "CUDA_VISIBLE_DEVICES",
    "CUDA_MODULE_LOADING", "CUDA_LAUNCH_BLOCKING", "LD_LIBRARY_PATH", "PYTHONPATH",
    "PYTHONUNBUFFERED", "PYTORCH_CUDA_ALLOC_CONF", "TRITON_CACHE_DIR", "XDG_CACHE_HOME",
    "SSL_CERT_FILE", "REQUESTS_CA_BUNDLE", "PIP_CACHE_DIR",
}
SECRET_NAME_RE = re.compile(r"TOKEN|SECRET|PASSWORD|API_KEY|AUTHORIZATION|CREDENTIAL", re.I)


def scrubbed_child_environment(extra=None):
    merged = {name: value for name, value in os.environ.items()
              if name in CHILD_ENV_ALLOWLIST and not SECRET_NAME_RE.search(name)}
    merged.setdefault("PATH", "/usr/local/bin:/usr/bin:/bin")
    merged.setdefault("HOME", "/root")
    merged["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    merged["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    if extra:
        rejected = [n for n in extra if n not in CHILD_ENV_ALLOWLIST or SECRET_NAME_RE.search(n)]
        if rejected:
            raise RuntimeError(f"Refusing secret-like child environment entries: {rejected}")
        merged.update(extra)
    return merged


def run(command, cwd=None, env=None):
    print("$ " + " ".join(str(part) for part in command))
    return subprocess.run(command, cwd=cwd, env=scrubbed_child_environment(env), check=True)


def sha256_file(path, chunk_bytes=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(block)
    return digest.hexdigest()


import torch  # noqa: E402

frames = GEOMETRY["length"]
if (frames - 5) % 17 != 0:
    raise RuntimeError(f"H3 frame count must satisfy n % 17 == 5; got {frames}")
if GEOMETRY["width"] % 32 or GEOMETRY["height"] % 32:
    raise RuntimeError(f"H3 dimensions must be divisible by 32: {GEOMETRY}")
LATENT_T = ((frames - 5) // 17) * 5 + 2
EXPECTED_LATENT_SHAPES = [
    [1, 24, LATENT_T, GEOMETRY["height"] // 16, GEOMETRY["width"] // 16],
    [1, 32, 2, round(frames / FPS * 40)],
]

# The blob's identity is its inputs. Recomputing the key here means a name match
# is itself proof that repo, revision, prompt and geometry all still agree.
CONDITIONING_KEY = "cond-" + hashlib.sha256(json.dumps({
    "repo": MODEL_REPO, "revision": MODEL_REVISION,
    "text_encoder_sha256": ENCODE_FILES[TEXT_ENCODER_FILE]["sha256"],
    "prompt_sha256": hashlib.sha256(PROMPT.encode("utf-8")).hexdigest(),
    **GEOMETRY,
}, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()[:32]

# Drive first: it is the only place a blob survives a recycled runtime.
blob_candidates = []
if CONDITIONING_BLOB.strip():
    blob_candidates.append(Path(CONDITIONING_BLOB.strip()))
blob_candidates += [
    DRIVE_CONDITIONING_STORE / f"{CONDITIONING_KEY}.pt",
    CONDITIONING_STORE / f"{CONDITIONING_KEY}.pt",
    CONTENT / "h3_encode_probe" / "conditioning" / f"{CONDITIONING_KEY}.pt",
]
RESOLVED_BLOB = next((p for p in blob_candidates if p.is_file()), None)
virtual_memory = psutil.virtual_memory()
disk = shutil.disk_usage("/content")
PROBE["stages"]["environment"] = {
    "measured_at": utc_now(),
    "run_id": RUN_ID,
    "host_ram_total_gib": round(virtual_memory.total / GIB, 3),
    "host_ram_available_gib": round(virtual_memory.available / GIB, 3),
    "cpu_count": os.cpu_count(),
    "disk_free_gib": round(disk.free / GIB, 3),
    "python": sys.version.split()[0],
    "torch": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "compute_capability": ".".join(str(x) for x in torch.cuda.get_device_capability(0))
    if torch.cuda.is_available() else None,
    "vram_total_mib": round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 2, 1)
    if torch.cuda.is_available() else None,
    "dit_choice": DIT_CHOICE,
    "dit": {**DIT, "gib": round(DIT["bytes"] / GIB, 3)},
    "unet_dtype": UNET_DTYPE,
    "unet_dtype_flag": UNET_DTYPE_FLAG,
    "rmsnorm_fp32": RMSNORM_FP32,
    "nan_trace": NAN_TRACE,
    "perf_probe": PERF_PROBE,
    "fp32_text_branch": FP32_TEXT_BRANCH,
    "turbo_lora": TURBO_LORA,
    "lora_strength": LORA_STRENGTH if LORA else None,
    "lora_low_vram": LORA_LOW_VRAM if LORA else None,
    "decode_to_mp4": DECODE_TO_MP4,
    "gear": GEAR,
    "geometry": GEOMETRY,
    "clip_seconds": CLIP_SECONDS,
    "gear_pixel_ratio": GEAR_PIXEL_RATIO,
    "steps": STEPS,
    "expected_latent_shapes": EXPECTED_LATENT_SHAPES,
    "conditioning_key": CONDITIONING_KEY,
    "drive_mounted": DRIVE_MOUNTED,
    "blob_searched": [str(p) for p in blob_candidates],
    "blob_resolved": str(RESOLVED_BLOB) if RESOLVED_BLOB else None,
}
save()

print(json.dumps(PROBE["stages"]["environment"], indent=2, ensure_ascii=False))
print()
print("=" * 72)
print(f"available RAM      : {virtual_memory.available / GIB:.2f} GiB")
print(f"DiT                : {DIT_CHOICE}  {DIT['bytes'] / GIB:.2f} GiB "
      f"(declares arch {DIT['gguf_arch']!r})")
print(f"unet dtype         : {UNET_DTYPE}  -> {UNET_DTYPE_FLAG or '(ComfyUI default)'}")
print(f"fp32 RMSNorm       : {RMSNORM_FP32}"
      f"{'  (acts only on fp16 input)' if RMSNORM_FP32 else ''}")
print(f"Turbo LoRA         : {TURBO_LORA}"
      + (f"  strength {LORA_STRENGTH}"
         f"{'  (merged, low_vram)' if LORA_LOW_VRAM else '  (run-time)'}" if LORA else ""))
if LORA and not 4 <= STEPS <= 8:
    print(f"  NOTE: STEPS = {STEPS}. The LoRA's documented range is 4-8; 4 is the "
          "minimum, 6-8 looks better, past 8 stops helping.")
if not LORA and STEPS < 20:
    print(f"  NOTE: STEPS = {STEPS} without the LoRA. The base model wants ~20; "
          "fewer will decode to something under-denoised.")
print(f"decode to MP4      : {DECODE_TO_MP4}")
print(f"fp32 text branch   : {FP32_TEXT_BRANCH}"
      f"{'  (condition_proj + token_refiner)' if FP32_TEXT_BRANCH else ''}")
print(f"NaN trace          : {NAN_TRACE}"
      f"{'  (diagnostic: use STEPS = 1)' if NAN_TRACE else ''}")
print(f"perf probe         : {PERF_PROBE}"
      f"{'  (cliff diagnosis: STEPS = 2 at gear 0.4 and 0.5)' if PERF_PROBE else ''}")
if NAN_TRACE and STEPS > 2:
    print(f"  NOTE: STEPS = {STEPS}. The trace stops at the first non-finite module, "
          "so one step answers it; more only costs time.")
print(f"disk free          : {disk.free / GIB:.1f} GiB")
print(f"gear               : {GEAR}  ({GEAR_PIXEL_RATIO}x the pixels of gear 0.3, "
      f"{CLIP_SECONDS} s -> {GEOMETRY['length']} frames)")
print(f"geometry / steps   : {GEOMETRY['width']}x{GEOMETRY['height']} / "
      f"{frames} frames / {STEPS} steps")
print(f"conditioning key   : {CONDITIONING_KEY}")
print(f"Drive mounted      : {DRIVE_MOUNTED}")
print(f"blob               : {RESOLVED_BLOB or 'NOT FOUND'}")
print(f"license gate open  : {license_gate_open()}")
print("=" * 72)
print("Reference from the encode stage: peak 9.17 GiB against 11.18 GiB available.")
print("The encoder could be streamed because it is read once; the DiT is read on")
print("all 20 steps, so it has to fit rather than stream.")
if RESOLVED_BLOB is None:
    print()
    if REGENERATE_CONDITIONING_IF_MISSING:
        print(f"No blob for gear {GEAR}. Stage 2 will regenerate it: download the")
        print("14.61 GiB encoder and the 4.85 GiB VAE, encode once, and copy the 5 MiB")
        print("result to Drive so a recycled runtime never costs that again.")
        print()
        print("Heads-up if you plan to sample SEVERAL gears: each gear encoded from")
        print("here pays the 19.46 GiB download and the 66 s encoder load again. The")
        print("encode itself is seconds -- batching every gear against one encoder")
        print("load is the cheap way, so consider encoding them in one session.")
    else:
        print("No blob found and REGENERATE_CONDITIONING_IF_MISSING is False.")
        print("Set it True, or point CONDITIONING_BLOB at a copy.")


## Stage 1 -- ComfyUI + ComfyUI-GGUF + Turbo nodes (all pinned) + control nodes

Pinned commits, a guarded one-line architecture patch, and a small control-node
pack (save/load conditioning and latents, probes). Torch is never reinstalled.


In [ ]:
# Stage 1 - ComfyUI + ComfyUI-GGUF at pinned commits, the architecture patch,
# and the two nodes this stage needs.
import importlib.metadata as importlib_metadata
import tempfile

setup_started = time.perf_counter()
PROTECTED = {"torch", "torchvision", "torchaudio", "triton"}


def canonical(name):
    return re.sub(r"[-_.]+", "-", str(name)).lower()


def is_protected(name):
    return canonical(name) in PROTECTED or canonical(name).startswith("nvidia-")


def protected_snapshot():
    snapshot = {}
    for distribution in importlib_metadata.distributions():
        raw = distribution.metadata.get("Name")
        if raw and is_protected(raw):
            snapshot[canonical(raw)] = str(distribution.version)
    return dict(sorted(snapshot.items()))


def ensure_checkout(repo, destination, commit, allow_dirty=False):
    destination = Path(destination)
    if not destination.exists():
        run(["git", "clone", "--filter=blob:none", repo, str(destination)])
    if not allow_dirty:
        dirty = subprocess.check_output(
            ["git", "-C", str(destination), "status", "--porcelain", "--untracked-files=no"],
            text=True, env=scrubbed_child_environment()).strip()
        if dirty:
            raise RuntimeError(f"Refusing a dirty checkout: {destination}: {dirty[:1000]}")
    head = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"],
                                   text=True, env=scrubbed_child_environment()).strip()
    if head != commit:
        run(["git", "-C", str(destination), "fetch", "origin", commit])
        run(["git", "-C", str(destination), "checkout", "--force", "--detach", commit])
    head = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"],
                                   text=True, env=scrubbed_child_environment()).strip()
    if head != commit:
        raise RuntimeError(f"Checkout verification failed: HEAD={head}")


def pip_install_without_touching_torch(requirements_path, label):
    """Resolve first, refuse the plan if it would move torch, then install exactly."""
    before = protected_snapshot()
    handle, name = tempfile.mkstemp(prefix=f"h3-{label}-", suffix=".json")
    os.close(handle)
    report_path = Path(name)
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "--dry-run",
             "--report", str(report_path), "-r", str(requirements_path)])
        plan = json.loads(report_path.read_text(encoding="utf-8"))
    finally:
        report_path.unlink(missing_ok=True)
    actions = []
    for item in plan.get("install", []):
        metadata = item.get("metadata", {})
        pkg, version = canonical(metadata.get("name", "")), str(metadata.get("version", ""))
        if not re.fullmatch(r"[a-z0-9][a-z0-9-]*", pkg) or not version:
            raise RuntimeError(f"Unsafe package in {label} plan: {pkg!r} {version!r}")
        actions.append({"name": pkg, "spec": f"{pkg}=={version}"})
    blocked = [a for a in actions if is_protected(a["name"])]
    if blocked:
        raise RuntimeError(
            f"Refusing the {label} plan: it would mutate protected runtime distributions "
            f"{[a['spec'] for a in blocked]}. Every measurement in this project assumes "
            "the stock torch/triton stack.")
    if actions:
        run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             *[a["spec"] for a in actions]])
    after = protected_snapshot()
    if before != after:
        raise RuntimeError(f"Protected distributions changed during {label}: {before} -> {after}")
    return {"installed": [a["spec"] for a in actions], "protected_unchanged": True}


ensure_checkout(COMFY_REPO, COMFY_DIR, COMFY_COMMIT)
comfy_install = pip_install_without_touching_torch(COMFY_DIR / "requirements.txt", "comfyui")

# ComfyUI-GGUF is patched below, so its checkout is allowed to be dirty on re-runs.
ensure_checkout(GGUF_REPO, GGUF_NODE_DIR, GGUF_COMMIT, allow_dirty=True)
gguf_install = pip_install_without_touching_torch(GGUF_NODE_DIR / "requirements.txt", "gguf")

# The Turbo LoRA needs the author's nodes, not LoraLoaderModelOnly: on a pruned
# base the adaln update lives in a collapsed time-embedding space and has to be
# re-injected at run time. Pinned like everything else. The repo also carries a
# node.zip, which nothing imports -- the code is the single __init__.py.
turbo_node = {"requested": TURBO_LORA}
if LORA:
    ensure_checkout(TURBO_NODE_REPO, TURBO_NODE_DIR, TURBO_NODE_COMMIT, allow_dirty=True)
    grid = TURBO_NODE_DIR / "h3_silu_temb_grid.safetensors"
    turbo_node["commit"] = TURBO_NODE_COMMIT
    turbo_node["silu_temb_grid_present"] = grid.is_file()
    turbo_node["silu_temb_grid_bytes"] = grid.stat().st_size if grid.is_file() else None
    if not grid.is_file():
        raise RuntimeError(
            "h3_silu_temb_grid.safetensors is missing from the Turbo node checkout. "
            "It is what re-injects the adaln update on a pruned base, so the LoRA "
            "would apply only partially and silently.")
    print(f"Turbo nodes: {TURBO_NODE_COMMIT[:12]}, "
          f"silu_temb grid {turbo_node['silu_temb_grid_bytes']} bytes")
else:
    print("Turbo nodes: skipped (TURBO_LORA is off)")
print(f"installed comfyui:{len(comfy_install['installed'])} gguf:{len(gguf_install['installed'])} "
      "packages, torch untouched")

# --- The architecture patch -------------------------------------------------
# ComfyUI-GGUF validates a UNET GGUF's architecture against a hardcoded set in
# loader.py. minimax_h3 is not in it, so every H3 DiT GGUF is rejected before a
# tensor is read:  ValueError: Unexpected architecture type in GGUF file
# The fix is one string in one set. Applied as a guarded patch: the anchor must
# appear exactly once, and the before/after hashes and diff are recorded.
# (The in-memory variant of this idea comes from jlucasmcrell/ComfyUI-H3-Multishot,
# whose 80 KB of multishot orchestration this stage does not need.)
loader_path = GGUF_NODE_DIR / "loader.py"
loader_before = loader_path.read_text(encoding="utf-8")
arch_patch = {"anchor": GGUF_LOADER_ANCHOR,
              "sha256_before": hashlib.sha256(loader_before.encode()).hexdigest()}
if GGUF_LOADER_PATCHED in loader_before:
    arch_patch["state"] = "already_patched"
elif loader_before.count(GGUF_LOADER_ANCHOR) == 1:
    loader_path.write_text(loader_before.replace(GGUF_LOADER_ANCHOR, GGUF_LOADER_PATCHED, 1),
                           encoding="utf-8")
    arch_patch["state"] = "patched"
else:
    raise RuntimeError(
        f"Architecture patch anchor {GGUF_LOADER_ANCHOR!r} appears "
        f"{loader_before.count(GGUF_LOADER_ANCHOR)} times in {loader_path}; expected exactly "
        "once. ComfyUI-GGUF's layout changed -- do not guess, inspect IMG_ARCH_LIST by hand.")
loader_after = loader_path.read_text(encoding="utf-8")
arch_patch["sha256_after"] = hashlib.sha256(loader_after.encode()).hexdigest()
arch_patch["minimax_h3_present"] = "minimax_h3" in loader_after
arch_patch["img_arch_list_line"] = next(
    (line.strip() for line in loader_after.splitlines() if line.startswith("IMG_ARCH_LIST")), None)
if not arch_patch["minimax_h3_present"]:
    raise RuntimeError("Architecture patch did not take effect")
arch_patch["diff"] = subprocess.check_output(
    ["git", "-C", str(GGUF_NODE_DIR), "diff", "--", "loader.py"],
    text=True, env=scrubbed_child_environment())
(WORK / "gguf_arch.patch").write_text(arch_patch["diff"], encoding="utf-8")
print(f"architecture patch: {arch_patch['state']}")
print(f"  {arch_patch['img_arch_list_line']}")

custom_nodes = COMFY_DIR / "custom_nodes"
# The allowlist is the point of this check: anything here was put here on purpose,
# at a pinned commit. The Turbo pack joins it only when the LoRA is actually in use,
# so turning the LoRA off still means it is not silently sitting in the runtime.
allowed = {"ComfyUI-GGUF", "H3BenchmarkControl", "__pycache__",
           "example_node.py.example", "websocket_image_save.py"}
if LORA:
    allowed.add(TURBO_NODE_DIR.name)
unexpected = [p.name for p in custom_nodes.iterdir() if p.name not in allowed]
if unexpected:
    raise RuntimeError(f"Unexpected custom nodes in a controlled runtime: {unexpected}")

CONTROL_NODE_DIR.mkdir(parents=True, exist_ok=True)
(CONTROL_NODE_DIR / "__init__.py").write_text(r'''
import json
import logging
import re
import sys
import time
from pathlib import Path

import torch

H3_TORCH_NESTED = "__h3_torch_nested__"
H3_COMFY_NESTED = "__h3_comfy_nested__"


def _is_comfy_nested(obj):
    """comfy.nested_tensor.NestedTensor, duck-typed.

    MiniMaxH3ImageToVideo returns its LATENT as this class rather than a torch
    nested tensor, so isinstance(obj, torch.Tensor) does not catch it.
    """
    return (not isinstance(obj, torch.Tensor)
            and bool(getattr(obj, "is_nested", False))
            and hasattr(obj, "unbind"))


def _restore(obj):
    if isinstance(obj, dict):
        keys = set(obj.keys())
        if keys == {H3_TORCH_NESTED}:
            return torch.nested.nested_tensor(list(obj[H3_TORCH_NESTED]))
        if keys == {H3_COMFY_NESTED}:
            from comfy.nested_tensor import NestedTensor
            return NestedTensor(list(obj[H3_COMFY_NESTED]))
        return {key: _restore(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_restore(value) for value in obj)
    if isinstance(obj, list):
        return [_restore(value) for value in obj]
    return obj


def _summarize(obj):
    if _is_comfy_nested(obj):
        return {"kind": "comfy_nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {"kind": "nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
        return {"kind": "tensor", "shape": list(obj.shape), "dtype": str(obj.dtype),
                "device": str(obj.device),
                "finite": bool(torch.isfinite(obj).all().item())
                if obj.is_floating_point() else None}
    if isinstance(obj, dict):
        return {key: _summarize(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_summarize(value) for value in obj]
    return {"kind": type(obj).__name__, "repr": repr(obj)[:200]}


def _detach_to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {H3_TORCH_NESTED: [p.detach().cpu() for p in obj.unbind()]}
        return obj.detach().cpu()
    if _is_comfy_nested(obj):
        return {H3_COMFY_NESTED: [p.detach().cpu() for p in obj.unbind()]}
    if isinstance(obj, dict):
        return {key: _detach_to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_detach_to_cpu(value) for value in obj)
    if isinstance(obj, list):
        return [_detach_to_cpu(value) for value in obj]
    return obj


class H3SaveConditioning:
    """Write the encoder's output so later runs never load the encoder.

    Present here only for the regeneration path in Stage 2: the sampling graph
    itself must not contain a CLIPLoader.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "conditioning": ("CONDITIONING",), "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ()
    FUNCTION = "save"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save(self, conditioning, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = {"schema_version": 1,
                   "conditioning": _detach_to_cpu(conditioning),
                   "samples": _detach_to_cpu(samples)}
        destination = directory / f"{key}.pt"
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        (directory / f"{key}.summary.json").write_text(json.dumps(
            {"key": key, "bytes": destination.stat().st_size,
             "conditioning": _summarize(payload["conditioning"]),
             "samples": _summarize(payload["samples"])}, indent=2), encoding="utf-8")
        logging.info("H3_SAVE_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return {}


class H3LoadConditioning:
    """Return a precomputed CONDITIONING/LATENT pair without a CLIPLoader.

    This is what keeps the 14.61 GiB text encoder out of the sampling graph.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"}),
        }}
    RETURN_TYPES = ("CONDITIONING", "LATENT")
    RETURN_NAMES = ("conditioning", "samples")
    FUNCTION = "load"
    CATEGORY = "H3 benchmark/control"

    def load(self, store_dir, key):
        source = Path(store_dir) / f"{key}.pt"
        if not source.is_file():
            raise RuntimeError(f"Precomputed conditioning is missing: {source}")
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError(f"Unsupported schema: {payload.get('schema_version')}")
        logging.info("H3_LOAD_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": source.stat().st_size}, separators=(",", ":")))
        return (_restore(payload["conditioning"]), _restore(payload["samples"]))


class H3LatentProbe:
    """Record the sampled latent without decoding it.

    Decoding needs the VAEs, and this stage is about the DiT alone.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "filename_prefix": ("STRING", {"default": "h3_probe/probe"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save_summary"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save_summary(self, samples, filename_prefix):
        import folder_paths
        summary = _summarize(samples)
        packed = samples.get("samples")
        parts = packed.unbind() if (_is_comfy_nested(packed)
                                    or getattr(packed, "is_nested", False)) else [packed]
        finite = [bool(torch.isfinite(p).all().item()) for p in parts
                  if isinstance(p, torch.Tensor)]
        # Write the report BEFORE raising: on a non-finite run the trace is the
        # whole point of the run, and an exception here must not discard it.
        output_path = Path(folder_paths.get_output_directory()) / f"{filename_prefix}.latent_summary.json"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {"summary": summary,
                   "shapes": [list(p.shape) for p in parts if isinstance(p, torch.Tensor)],
                   "finite": finite,
                   "nan_trace": _trace_report(),
                   "perf": _perf_report()}
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        logging.info("H3_LATENT_PROBE %s", json.dumps(payload["shapes"], separators=(",", ":")))
        if payload["nan_trace"] is not None:
            logging.info("H3_NAN_TRACE %s", json.dumps(
                payload["nan_trace"], separators=(",", ":")))
        if payload["perf"] is not None:
            logging.info("H3_PERF_REPORT %s", json.dumps(
                payload["perf"], separators=(",", ":")))
        if not finite or not all(finite):
            raise RuntimeError(f"Sampled latent contains non-finite values: {summary}")
        return (samples,)


# --- locating the first non-finite value -------------------------------------
# fp16 sampling produces a non-finite latent and upcasting RMSNorm did not fix it,
# so the overflow is somewhere else. Rather than patch by guesswork, hook every
# submodule of the DiT and record the FIRST one whose output goes non-finite while
# its input was still finite. That module is where the value is created; everything
# after it is downstream noise, so recording stops there.
#
# Module-level hooks are enough to answer the question that matters. The fused rope
# path is a function call inside Attention, not a module, so "Attention output bad,
# Attention input fine" localises it to the fused kernel without hooking the kernel.
#
# Cost: one GPU sync per module until the first bad one. At one step that is fine;
# it is not something to leave on for a real run.
H3_NAN_TRACE = {"armed": False, "done": False, "order": 0, "trail": [],
                "trail_len": 16, "culprit": None, "hooks": 0, "error": None,
                "skipped": 0, "errors": []}


def _trace_first_tensor(obj, depth=0):
    if isinstance(obj, torch.Tensor):
        return obj if obj.is_floating_point() else None
    if depth > 2:
        return None
    if isinstance(obj, (list, tuple)):
        for item in obj:
            found = _trace_first_tensor(item, depth + 1)
            if found is not None:
                return found
        return None
    if isinstance(obj, dict):
        return _trace_first_tensor(list(obj.values()), depth + 1)
    inner = getattr(obj, "tensors", None)           # comfy.nested_tensor.NestedTensor
    if isinstance(inner, (list, tuple)):
        return _trace_first_tensor(inner, depth + 1)
    return None


def _trace_stats(tensor):
    """min and max, one sync, no full-size temporary.

    The first version built an isfinite mask and an abs() copy, which is three
    extra tensors the size of the activation. On a 44232x5376 fp16 activation that
    is about a gigabyte, and it pushed a 14.9 GiB card into OOM mid-trace. aminmax
    is a reduction: it allocates two scalars.

    min/max propagate NaN, and inf survives them, so two numbers answer all three
    questions. The cost is that absmax is the max over ALL entries rather than over
    the finite ones -- which only matters on the tensor that already went bad.
    """
    low, high = torch.aminmax(tensor.detach())
    low, high = torch.stack([low.to(torch.float32),
                             high.to(torch.float32)]).tolist()
    has_nan = low != low or high != high          # NaN is the only value unequal to itself
    infinity = float("inf")
    has_inf = not has_nan and (low == -infinity or high == infinity)
    finite = not has_nan and not has_inf
    return {"finite": finite, "has_nan": has_nan, "has_inf": has_inf,
            "absmax": round(max(abs(low), abs(high)), 4) if finite else None,
            "min": round(low, 4) if finite else None,
            "max": round(high, 4) if finite else None}


def _trace_hook(name):
    def hook(module, args, output):
        state = H3_NAN_TRACE
        if state["done"]:
            return
        try:
            out = _trace_first_tensor(output)
            if out is None or out.numel() == 0:
                return
            state["order"] += 1
            stats = _trace_stats(out)
            record = {"order": state["order"], "module": name or "<root>",
                      "cls": type(module).__name__, "dtype": str(out.dtype),
                      "out": stats}
            state["trail"].append(record)
            if len(state["trail"]) > state["trail_len"]:
                state["trail"].pop(0)
            if not stats["finite"]:
                inp = _trace_first_tensor(args)
                record["in"] = (_trace_stats(inp) if inp is not None and inp.numel()
                                else None)
                state["culprit"] = record
                state["done"] = True    # everything downstream is just contaminated
        except Exception as exc:
            # Skip this module and keep going. An earlier version stopped the whole
            # trace on the first error, so one OOM inside the instrumentation threw
            # away the answer the run existed to produce.
            state["errors"] = state.get("errors", [])
            if len(state["errors"]) < 8:
                state["errors"].append(f"{name}: {type(exc).__name__}: "
                                       f"{str(exc).splitlines()[0][:160]}")
            state["skipped"] = state.get("skipped", 0) + 1
    return hook


def _trace_report():
    state = H3_NAN_TRACE
    if not state["armed"]:
        return None
    return {"armed": True, "hooks": state["hooks"], "modules_seen": state["order"],
            "culprit": state["culprit"], "trail": state["trail"],
            "error": state["error"], "skipped": state.get("skipped", 0),
            "errors": state.get("errors", []),
            "note": ("culprit is the first module whose output went non-finite; "
                     "in.finite=true there means the value is created inside it")}


class H3SaveLatent:
    """Persist the sampled AV latent so decoding can happen in another process.

    The DiT is 10.77 GiB and the video VAE is 4.85 GiB. Decoding in the same
    ComfyUI that just sampled would put both through a 12.7 GiB host, which is
    the sum the three-stage split exists to avoid. So the latent goes to disk and
    a fresh process picks it up.

    This runs BEFORE H3LatentProbe, so a non-finite latent is still written --
    a failed run is exactly when you want the artefact to look at.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": "/content/h3_sample_probe/latents"}),
            "key": ("STRING", {"default": "latent"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save(self, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        path = directory / f"{key}.pt"
        torch.save({"schema_version": 1, "key": key,
                    "samples": _detach_to_cpu(samples)}, path)
        logging.info("H3_SAVE_LATENT %s", json.dumps(
            {"key": key, "path": str(path), "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3LoadLatent:
    """Read back what H3SaveLatent wrote, in a process with no DiT in it."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": "/content/h3_sample_probe/latents"}),
            "key": ("STRING", {"default": "latent"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "load"
    CATEGORY = "H3 benchmark/control"

    def load(self, store_dir, key):
        path = Path(store_dir) / f"{key}.pt"
        if not path.is_file():
            raise RuntimeError(f"No latent at {path}. Run the sampling stage first.")
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError(f"Unsupported schema: {payload.get('schema_version')}")
        samples = _restore(payload["samples"])
        logging.info("H3_LOAD_LATENT %s", json.dumps(
            {"key": key, "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3Fp32TextBranch:
    """Run the text branch in fp32 and pass the MODEL through.

    condition_proj is where fp16 dies: measured input absmax 15424, output inf.
    Only the input needs upcasting -- comfy's cast_bias_weight and ComfyUI-GGUF's
    dequant both target the input dtype, so token_refiner follows into fp32 by
    itself and returns through final_norm (a RMSNorm), which puts the values back
    at O(1) before they are written into the fp16 sequence buffer.

    Video and audio already get this treatment: ComfyUI constructs video_patch_proj
    and audio_patch_proj with dtype=torch.float32. This extends it to text.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"model": ("MODEL",)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "apply"
    CATEGORY = "H3 benchmark/control"

    def apply(self, model):
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        proj = getattr(target, "condition_proj", None)
        if proj is None:
            logging.warning("H3_FP32_TEXT_BRANCH no condition_proj found; nothing patched")
            return (model,)
        if getattr(proj, "_h3_fp32_text_branch", False):
            logging.info("H3_FP32_TEXT_BRANCH already_applied")
            return (model,)
        original = proj.forward

        def forward(x, *args, **kwargs):
            # bf16 and fp32 already have the range; only fp16 needs this.
            if isinstance(x, torch.Tensor) and x.dtype == torch.float16:
                x = x.to(torch.float32)
            return original(x, *args, **kwargs)

        proj.forward = forward
        proj._h3_fp32_text_branch = True
        logging.info("H3_FP32_TEXT_BRANCH applied to condition_proj "
                     "(token_refiner follows the input dtype)")
        return (model,)


class H3NanTrace:
    """Arm the tracer on a MODEL and pass it straight through."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "model": ("MODEL",),
            "trail_len": ("INT", {"default": 16, "min": 1, "max": 256}),
        }}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "arm"
    CATEGORY = "H3 benchmark/control"

    def arm(self, model, trail_len):
        state = H3_NAN_TRACE
        state.update({"armed": True, "done": False, "order": 0, "trail": [],
                      "trail_len": int(trail_len), "culprit": None, "error": None,
                      "skipped": 0, "errors": []})
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        if target is None:
            state["error"] = "no diffusion_model on the patcher; nothing hooked"
            logging.warning("H3_NAN_TRACE_ARM %s", state["error"])
            return (model,)
        if getattr(target, "_h3_trace_hooked", False):
            logging.info("H3_NAN_TRACE_ARM already hooked, counters reset")
            return (model,)
        count = 0
        for name, module in target.named_modules():
            module.register_forward_hook(_trace_hook(name))
            count += 1
        target._h3_trace_hooked = True
        state["hooks"] = count
        logging.info("H3_NAN_TRACE_ARM hooked %d modules", count)
        return (model,)


# --- where a step's time goes: attention, MLP, or the allocator ---------------
# The cliff between gear 0.4 and 0.5 is a step in the activations, not a curve
# (per-pixel cost 233 -> 354), and shrinking the DiT by 2.5 GiB moved it by 2%.
# Three candidate mechanisms remain, and they leave different fingerprints in
# the same three measurements:
#   1. per-class CUDA time  -- an attention kernel fallback inflates the
#                              attention classes only, superlinearly
#   2. SDPA call census     -- a chunked/split path changes the call count and
#                              shapes between gears; zero calls means attention
#                              is not going through F.sdpa at all
#   3. allocator counters   -- thrashing shows as alloc retries and device
#                              alloc/free churn, and it inflates EVERY class
# Cost: two CUDA events per hooked call and ONE synchronize per model forward.
# A step is minutes; the probe is milliseconds. Probed timings stay measurements.
H3_PERF = {"armed": False, "forwards": 0, "hooks": 0, "events": [], "sdpa": {},
           "sdpa_events": [], "backend_cache": {}, "steps": [], "error": None,
           "pre_counters": None, "pre_t": None, "root_events": None,
           "rebound": 0, "allocator_backend": None}

_PERF_CLASS_RE = re.compile(r"attention|attn|mlp|feedforward|ffn", re.I)
_SDP_BACKEND_NAMES = {-1: "ERROR", 0: "MATH", 1: "FLASH_ATTENTION",
                      2: "EFFICIENT_ATTENTION", 3: "CUDNN_ATTENTION",
                      4: "OVERRIDEABLE"}


def _perf_counters():
    """Allocator event counters, for deltas across one forward.

    No reset_peak_memory_stats: ComfyUI reads the same accounting for its own
    load/unload decisions, so peaks are reported monotonically and the per-step
    signal lives in these monotone counters instead.
    """
    try:
        stats = torch.cuda.memory_stats()
    except Exception:
        return {}
    return {key: int(stats.get(key, 0)) for key in (
        "num_alloc_retries", "num_ooms", "num_device_alloc", "num_device_free")}


def _perf_event():
    event = torch.cuda.Event(enable_timing=True)
    event.record()
    return event


def _perf_pre_module(module, args):
    if not H3_PERF["armed"]:
        return
    try:
        stack = getattr(module, "_h3_perf_stack", None)
        if stack is None:
            stack = []
            module._h3_perf_stack = stack
        stack.append(_perf_event())
    except Exception:
        pass


def _perf_post_module(module, args, output):
    state = H3_PERF
    if not state["armed"]:
        return
    try:
        stack = getattr(module, "_h3_perf_stack", None)
        if not stack:
            return
        begin = stack.pop()
        state["events"].append((type(module).__name__, begin, _perf_event()))
    except Exception:
        pass


def _perf_sdpa_wrapper(original):
    def wrapped(query, key, value, *args, **kwargs):
        state = H3_PERF
        if (not state["armed"] or not isinstance(query, torch.Tensor)
                or not query.is_cuda):
            return original(query, key, value, *args, **kwargs)
        begin = signature = None
        try:
            mask = kwargs.get("attn_mask", args[0] if args else None)
            dropout = kwargs.get("dropout_p", args[1] if len(args) > 1 else 0.0)
            causal = bool(kwargs.get("is_causal",
                                     args[2] if len(args) > 2 else False))
            signature = (f"q{tuple(query.shape)} kv{tuple(key.shape)} "
                         f"{str(query.dtype).replace('torch.', '')} "
                         f"causal={causal} mask={mask is not None}")
            if signature not in state["backend_cache"]:
                # What the dispatcher WOULD pick for this exact call. Cached per
                # signature; the answer cannot change for identical inputs.
                try:
                    choice = torch._fused_sdp_choice(query, key, value, mask,
                                                     dropout, causal)
                    state["backend_cache"][signature] = _SDP_BACKEND_NAMES.get(
                        int(choice), f"UNKNOWN_{choice}")
                except Exception as exc:
                    state["backend_cache"][signature] = \
                        f"choice_unavailable:{type(exc).__name__}"
            state["sdpa"].setdefault(signature,
                                     {"calls": 0, "ms": 0.0})["calls"] += 1
            begin = _perf_event()
        except Exception:
            begin = None
        out = original(query, key, value, *args, **kwargs)
        if begin is not None:
            try:
                state["sdpa_events"].append((signature, begin, _perf_event()))
            except Exception:
                pass
        return out
    wrapped._h3_perf_original = original
    return wrapped


def _perf_install_sdpa():
    """Wrap F.scaled_dot_product_attention and rebind every module-level alias.

    Modules that did `from torch.nn.functional import scaled_dot_product_attention`
    hold their own reference, so patching the F attribute alone misses them. The
    scan replaces any module attribute that IS the original function object,
    which is semantically transparent. If the census still records zero calls,
    that is itself the answer: attention runs through a fused non-SDPA path.
    """
    original = torch.nn.functional.scaled_dot_product_attention
    if getattr(original, "_h3_perf_original", None) is not None:
        return 0                                   # already wrapped
    wrapped = _perf_sdpa_wrapper(original)
    rebound = 0
    for _name, module in list(sys.modules.items()):
        if module is None:
            continue
        try:
            for attr, value in list(vars(module).items()):
                if value is original:
                    setattr(module, attr, wrapped)
                    rebound += 1
        except Exception:
            continue
    return rebound


def _perf_pre_root(module, args):
    state = H3_PERF
    if not state["armed"]:
        return
    try:
        state["pre_counters"] = _perf_counters()
        state["pre_t"] = time.perf_counter()
        state["events"] = []
        state["sdpa_events"] = []
        state["root_events"] = (_perf_event(),
                                torch.cuda.Event(enable_timing=True))
    except Exception as exc:
        state["root_events"] = None
        state["error"] = f"pre_root {type(exc).__name__}: {str(exc)[:200]}"


def _perf_post_root(module, args, output):
    state = H3_PERF
    if not state["armed"] or state["root_events"] is None:
        return
    try:
        begin, end = state["root_events"]
        state["root_events"] = None
        end.record()
        torch.cuda.synchronize()               # the one sync this probe costs
        total_ms = begin.elapsed_time(end)
        per_class = {}
        for cls, b, e in state["events"]:
            entry = per_class.setdefault(cls, {"ms": 0.0, "calls": 0})
            entry["ms"] += b.elapsed_time(e)
            entry["calls"] += 1
        hooked_ms = sum(v["ms"] for v in per_class.values())
        for entry in per_class.values():
            entry["ms"] = round(entry["ms"], 1)
        sdpa_ms = 0.0
        for signature, b, e in state["sdpa_events"]:
            elapsed = b.elapsed_time(e)
            sdpa_ms += elapsed
            record = state["sdpa"].setdefault(signature,
                                              {"calls": 0, "ms": 0.0})
            record["ms"] += elapsed
        post = _perf_counters()
        pre = state["pre_counters"] or {}
        try:
            stats = torch.cuda.memory_stats()
            memory = {"allocated_mib": round(
                          stats.get("allocated_bytes.all.current", 0) / 2**20),
                      "reserved_mib": round(
                          stats.get("reserved_bytes.all.current", 0) / 2**20),
                      "allocated_peak_mib": round(
                          stats.get("allocated_bytes.all.peak", 0) / 2**20)}
        except Exception:
            memory = {}
        state["forwards"] += 1
        step = {"forward": state["forwards"],
                "wall_s": (round(time.perf_counter() - state["pre_t"], 2)
                           if state["pre_t"] else None),
                "cuda_total_ms": round(total_ms, 1),
                "per_class_ms": per_class,
                "unhooked_ms": round(total_ms - hooked_ms, 1),
                "sdpa_calls": len(state["sdpa_events"]),
                "sdpa_ms": round(sdpa_ms, 1),
                "alloc_delta": {key: post.get(key, 0) - pre.get(key, 0)
                                for key in post}}
        step.update(memory)
        state["steps"].append(step)
        if len(state["steps"]) > 64:
            state["steps"].pop(0)
        state["events"] = []
        state["sdpa_events"] = []
        logging.info("H3_PERF_STEP %s", json.dumps(step, separators=(",", ":")))
    except Exception as exc:
        state["error"] = f"post_root {type(exc).__name__}: {str(exc)[:200]}"


def _perf_report():
    state = H3_PERF
    if not state["armed"]:
        return None
    census = [{"sig": sig, "backend": state["backend_cache"].get(sig),
               "calls": entry["calls"], "ms": round(entry["ms"], 1)}
              for sig, entry in sorted(state["sdpa"].items())]
    return {"armed": True, "hooks": state["hooks"],
            "forwards": state["forwards"],
            "sdpa_refs_rebound": state["rebound"],
            "allocator_backend": state["allocator_backend"],
            "sdpa_census": census, "steps": state["steps"],
            "error": state["error"],
            "note": ("per_class_ms is CUDA time by module class; a kernel "
                     "fallback inflates attention only, allocator thrashing "
                     "raises alloc_delta counters, a split path changes sdpa "
                     "call count/shapes, sdpa_calls=0 means a fused non-SDPA "
                     "path")}


class H3PerfProbe:
    """Arm the per-step performance probe on a MODEL and pass it through.

    Same shape as H3NanTrace: a pass-through on MODEL that instruments the
    diffusion model underneath. The three measurements together separate the
    cliff candidates; see the comment block above H3_PERF.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"model": ("MODEL",)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "arm"
    CATEGORY = "H3 benchmark/control"

    def arm(self, model):
        state = H3_PERF
        state.update({"armed": True, "forwards": 0, "events": [], "sdpa": {},
                      "sdpa_events": [], "backend_cache": {}, "steps": [],
                      "error": None, "pre_counters": None, "pre_t": None,
                      "root_events": None})
        try:
            state["allocator_backend"] = torch.cuda.get_allocator_backend()
        except Exception:
            state["allocator_backend"] = "unknown"
        state["rebound"] = _perf_install_sdpa()
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        if target is None:
            state["error"] = "no diffusion_model on the patcher; nothing hooked"
            logging.warning("H3_PERF_ARM %s", state["error"])
            return (model,)
        if getattr(target, "_h3_perf_hooked", False):
            logging.info("H3_PERF_ARM already hooked, counters reset")
            return (model,)
        target.register_forward_pre_hook(_perf_pre_root)
        target.register_forward_hook(_perf_post_root)
        selected = []
        classes = set()
        for name, module in target.named_modules():
            if not name or not _PERF_CLASS_RE.search(type(module).__name__):
                continue
            # named_modules yields parents before children; skipping descendants
            # of an already-timed module keeps the per-class sums disjoint.
            if any(name.startswith(prefix + ".") for prefix in selected):
                continue
            selected.append(name)
            classes.add(type(module).__name__)
            module.register_forward_pre_hook(_perf_pre_module)
            module.register_forward_hook(_perf_post_module)
        target._h3_perf_hooked = True
        state["hooks"] = len(selected)
        logging.info(
            "H3_PERF_ARM hooked %d modules (classes %s), rebound %d sdpa refs, "
            "allocator %s", len(selected), ",".join(sorted(classes)),
            state["rebound"], state["allocator_backend"])
        return (model,)



H3_RMSNORM_FP32 = __RMSNORM_FP32__


def _install_fp32_rmsnorm():
    """Run RMSNorm in fp32 without giving up fp16 tensor cores elsewhere.

    ComfyUI-GGUF never overrides RMSNorm, so replacing the method on
    disable_weight_init reaches every RMSNorm the GGUF path uses. The fused
    q_norm/k_norm inside Attention calls comfy_kitchen directly and is not covered.
    """
    import comfy.ops as ops

    target = ops.disable_weight_init.RMSNorm
    if getattr(target, "_h3_fp32_rmsnorm", False):
        return "already_applied"

    def forward_comfy_cast_weights(self, input):
        if self.weight is not None:
            weight, bias, offload_stream = ops.cast_bias_weight(
                self, input, offloadable=True)
        else:
            weight = bias = offload_stream = None
        if input.dtype == torch.float16:
            # bf16 and fp32 already have the exponent range; only fp16 needs this.
            x = torch.nn.functional.rms_norm(
                input.float(), self.normalized_shape,
                weight.float() if weight is not None else None,
                self.eps).to(input.dtype)
        else:
            x = torch.nn.functional.rms_norm(
                input, self.normalized_shape, weight, self.eps)
        uncast = getattr(ops, "uncast_bias_weight", None)
        if uncast is not None:
            uncast(self, weight, bias, offload_stream)
        return x

    target.forward_comfy_cast_weights = forward_comfy_cast_weights
    target._h3_fp32_rmsnorm = True
    return "applied"


if H3_RMSNORM_FP32:
    try:
        logging.info("H3_FP32_RMSNORM %s (fused q/k rope path is NOT covered)",
                     _install_fp32_rmsnorm())
    except Exception as exc:                                  # never fatal
        logging.warning("H3_FP32_RMSNORM failed: %s: %s", type(exc).__name__, exc)


NODE_CLASS_MAPPINGS = {
    "H3SaveConditioning": H3SaveConditioning,
    "H3LoadConditioning": H3LoadConditioning,
    "H3LatentProbe": H3LatentProbe,
    "H3NanTrace": H3NanTrace,
    "H3PerfProbe": H3PerfProbe,
    "H3Fp32TextBranch": H3Fp32TextBranch,
    "H3SaveLatent": H3SaveLatent,
    "H3LoadLatent": H3LoadLatent,
}
NODE_DISPLAY_NAME_MAPPINGS = {
    "H3SaveConditioning": "H3 Save Precomputed Conditioning",
    "H3LoadConditioning": "H3 Load Precomputed Conditioning",
    "H3LatentProbe": "H3 Sampled Latent Probe",
    "H3NanTrace": "H3 First Non-Finite Module Trace",
    "H3PerfProbe": "H3 Per-Step Performance Probe",
    "H3Fp32TextBranch": "H3 fp32 Text Branch",
    "H3SaveLatent": "H3 Save Sampled Latent",
    "H3LoadLatent": "H3 Load Sampled Latent",
}
'''.replace("__RMSNORM_FP32__", repr(bool(RMSNORM_FP32))), encoding="utf-8")

# --- ComfyUI process control ------------------------------------------------
# These live here rather than in Stage 3 because Stage 2 may need to start ComfyUI
# to regenerate the conditioning blob. An earlier version defined them in Stage 3
# and the regeneration path then failed with NameError.
import requests  # noqa: E402

COMFY_PROCESS = globals().get("COMFY_PROCESS")
COMFY_LOG_HANDLE = globals().get("COMFY_LOG_HANDLE")
FORBIDDEN_FLAGS = ("--use-sage-attention", "--highvram")


def stop_comfy(term_timeout_s=15):
    global COMFY_PROCESS, COMFY_LOG_HANDLE
    if COMFY_PROCESS is not None and COMFY_PROCESS.poll() is None:
        os.killpg(COMFY_PROCESS.pid, signal.SIGTERM)
        try:
            COMFY_PROCESS.wait(timeout=term_timeout_s)
        except subprocess.TimeoutExpired:
            os.killpg(COMFY_PROCESS.pid, signal.SIGKILL)
            COMFY_PROCESS.wait(timeout=15)
    COMFY_PROCESS = None
    if COMFY_LOG_HANDLE is not None:
        COMFY_LOG_HANDLE.close()
        COMFY_LOG_HANDLE = None


def supported_cli_flags():
    path = COMFY_DIR / "comfy" / "cli_args.py"
    text = path.read_text(encoding="utf-8") if path.is_file() else ""
    return set(re.findall(r'"(--[a-z0-9][a-z0-9-]*)"', text)) | \
        set(re.findall(r"'(--[a-z0-9][a-z0-9-]*)'", text))


def start_comfy(timeout_s=900, quiet=False, dtype_flag="from_config"):
    """dtype_flag="from_config" uses UNET_DTYPE_FLAG; None starts without one.

    The decode stage passes None: --force-fp16 is a statement about the DiT, and
    the audio VAE ships as fp32. Carrying a sampling-time dtype decision into a
    process that holds no DiT would be forcing a choice onto the wrong model.
    """
    global COMFY_PROCESS, COMFY_LOG_HANDLE, COMFY_FLAG_DECISION
    stop_comfy()
    log_offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
    COMFY_LOG_HANDLE = COMFY_LOG.open("a", encoding="utf-8", buffering=1)
    supported = supported_cli_flags()
    command = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
               "--disable-auto-launch", "--preview-method", "none", "--cache-classic"]
    requested, applied, dropped = [], [], []
    flag = UNET_DTYPE_FLAG if dtype_flag == "from_config" else dtype_flag
    if flag:
        requested.append(flag)
    if ENABLE_TRITON_BACKEND:
        requested.append("--enable-triton-backend")
    for flag in requested:
        (applied if flag in supported else dropped).append(flag)
        if flag in supported:
            command.append(flag)
    for forbidden in FORBIDDEN_FLAGS:
        if forbidden in command:
            raise RuntimeError(f"{forbidden} is forbidden here.")
    COMFY_FLAG_DECISION = {"applied": applied, "dropped_not_in_pinned_commit": dropped}
    print("$ " + " ".join(command))
    COMFY_PROCESS = subprocess.Popen(
        command, cwd=COMFY_DIR, env=scrubbed_child_environment({"PYTHONUNBUFFERED": "1"}),
        stdout=COMFY_LOG_HANDLE, stderr=subprocess.STDOUT, start_new_session=True)
    deadline, shown, last_error = time.time() + timeout_s, log_offset, None
    while time.time() < deadline:
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError("ComfyUI exited during startup:\n"
                               + COMFY_LOG.read_text(errors="replace")[-8000:])
        if COMFY_LOG.exists():
            with COMFY_LOG.open("rb") as reader:
                reader.seek(shown)
                chunk = reader.read().decode("utf-8", errors="replace")
                shown += len(chunk.encode("utf-8"))
            if not quiet:
                for line in chunk.splitlines():
                    if line.strip():
                        print("    " + line[:160])
        try:
            response = requests.get(f"{BASE_URL}/system_stats", timeout=5)
            if response.status_code == 200:
                return response.json(), log_offset
            last_error = f"HTTP {response.status_code}"
        except Exception as exc:
            last_error = type(exc).__name__
        time.sleep(2)
    raise RuntimeError(f"ComfyUI startup timeout: {last_error}")


def tail_comfy_log(offset):
    if COMFY_LOG_HANDLE is not None:
        COMFY_LOG_HANDLE.flush()
    if not COMFY_LOG.exists():
        return ""
    with COMFY_LOG.open("rb") as reader:
        reader.seek(offset)
        return reader.read().decode("utf-8", errors="replace")


# Configura extra_model_paths.yaml para ComfyUI ler modelos nativamente do Google Drive
if DRIVE_MOUNTED:
    extra_yaml_path = COMFY_DIR / "extra_model_paths.yaml"
    extra_yaml_content = f"""drive_comfy:
    base_path: {DRIVE_COMFY_DIR}
    checkpoints: models/checkpoints
    clip: models/text_encoders
    clip_vision: models/clip_vision
    configs: models/configs
    diffusion_models: models/diffusion_models
    embeddings: models/embeddings
    loras: models/loras
    upscale_models: models/upscale_models
    vae: models/vae
"""
    extra_yaml_path.write_text(extra_yaml_content, encoding="utf-8")
    print(f"📁 extra_model_paths.yaml configurado nativamente para {DRIVE_COMFY_DIR}")

PROBE["stages"]["setup"] = {
    "comfyui_commit": COMFY_COMMIT,
    "gguf_commit": GGUF_COMMIT,
    "turbo_node": turbo_node,
    "gguf_arch_patch": {k: v for k, v in arch_patch.items() if k != "diff"},
    "pip_comfyui": comfy_install,
    "pip_gguf": gguf_install,
    "control_nodes": ["H3SaveConditioning", "H3LoadConditioning", "H3LatentProbe",
                      "H3NanTrace", "H3PerfProbe", "H3Fp32TextBranch",
                      "H3SaveLatent", "H3LoadLatent"],
    "rmsnorm_fp32_armed": RMSNORM_FP32,
    "h3_multishot_pack": "not installed; only the one-line IMG_ARCH_LIST patch is needed",
    "elapsed_s": round(time.perf_counter() - setup_started, 1),
}
save()
print(json.dumps(PROBE["stages"]["setup"], indent=2, ensure_ascii=False)[:4000])


## 📦 Verificação e Download de Modelos no Google Drive

Esta célula verifica automaticamente os modelos necessários dentro da sua pasta **`MyDrive/ComfyUI/models/`** no Google Drive:
- Se o modelo **já estiver no seu Drive**, ele é detectado e **não será baixado novamente**.
- Se algum modelo estiver faltando, ele será baixado **direto para a pasta correta no Drive** (`diffusion_models`, `loras`, `vae`, `text_encoders`).
- Todos os modelos são vinculados diretamente ao ComfyUI via links simbólicos sem ocupar espaço no disco local da máquina.


In [ ]:
# =============================================================================
# Gerenciador e Downloader Inteligente de Modelos (Google Drive)
# =============================================================================
import os
import sys
import time
import shutil
from pathlib import Path
from google.colab import drive

# 1. Garantir montagem do Google Drive
DRIVE_MOUNT = Path('/content/drive')
if not (DRIVE_MOUNT / 'MyDrive').is_dir():
    print('📁 Conectando ao Google Drive...')
    drive.mount(str(DRIVE_MOUNT))

# 2. Localizar pasta ComfyUI no Drive
drive_candidates = [
    DRIVE_MOUNT / 'MyDrive' / 'ComfyUI',
    DRIVE_MOUNT / 'MyDrive' / 'ComfyUi',
    DRIVE_MOUNT / 'MyDrive' / 'comfyui',
]
DRIVE_COMFY_DIR = next((p for p in drive_candidates if p.is_dir()), DRIVE_MOUNT / 'MyDrive' / 'ComfyUI')
DRIVE_COMFY_MODELS = DRIVE_COMFY_DIR / 'models'
COMFY_LOCAL_MODELS = Path('/content/ComfyUI/models')

print('=' * 80)
print(f'📂 Pasta ComfyUI no Drive: {DRIVE_COMFY_DIR}')
print(f'📦 Pasta de Modelos       : {DRIVE_COMFY_MODELS}')
print('=' * 80)

# 3. Catálogo dos modelos necessários para o MiniMax H3
H3_MODELS = [
    {
        'id': 'dit',
        'name': 'MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf',
        'folder': 'diffusion_models',
        'alt_folders': ['unet'],
        'repo': 'Abiray/MiniMax-H3-Pruned-GGUF',
        'hf_filename': 'MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf',
        'bytes': 11564180576,
        'gib': 10.77,
        'desc': 'DiT Transformer (GGUF Q4_K_M - Vídeo + Áudio)'
    },
    {
        'id': 'lora',
        'name': 'minimax_h3_turbo_v4_step600_ema.safetensors',
        'folder': 'loras',
        'alt_folders': [],
        'repo': 'larryvrh/MiniMax-H3-Turbo-Lora',
        'hf_filename': 'minimax_h3_turbo_v4_step600_ema.safetensors',
        'bytes': 779849816,
        'gib': 0.73,
        'desc': 'Turbo LoRA (reduz para 6 passos)'
    },
    {
        'id': 'video_vae',
        'name': 'minimax_h3_video_vae_fp16.safetensors',
        'folder': 'vae',
        'alt_folders': [],
        'repo': 'Comfy-Org/MiniMax-H3',
        'hf_filename': 'vae/minimax_h3_video_vae_fp16.safetensors',
        'bytes': 5207808496,
        'gib': 4.85,
        'desc': 'Video VAE (fp16)'
    },
    {
        'id': 'audio_vae',
        'name': 'minimax_h3_audio_vae_fp32.safetensors',
        'folder': 'vae',
        'alt_folders': [],
        'repo': 'Comfy-Org/MiniMax-H3',
        'hf_filename': 'vae/minimax_h3_audio_vae_fp32.safetensors',
        'bytes': 605254808,
        'gib': 0.56,
        'desc': 'Audio VAE (fp32)'
    },
    {
        'id': 'text_encoder',
        'name': 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
        'folder': 'text_encoders',
        'alt_folders': ['clip'],
        'repo': 'Comfy-Org/MiniMax-H3',
        'hf_filename': 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
        'bytes': 15687142551,
        'gib': 14.61,
        'desc': 'Text Encoder Qwen3-VL (para codificar novos prompts)'
    },
]

# 4. Varredura no Drive
print('🔍 Verificando modelos existentes no Google Drive...')
to_download = []
found_models = {}

for m in H3_MODELS:
    target_dir = DRIVE_COMFY_MODELS / m['folder']
    target_dir.mkdir(parents=True, exist_ok=True)
    target_file = target_dir / m['name']
    
    # Procura no caminho primário e em pastas alternativas
    candidates = [target_file] + [DRIVE_COMFY_MODELS / alt / m['name'] for alt in m['alt_folders']]
    resolved = next((c for c in candidates if c.is_file() and c.stat().st_size == m['bytes']), None)
    
    if resolved:
        print(f"  [✅ JÁ EXISTE NO DRIVE] {m['desc']}")
        print(f"      Local: {resolved.relative_to(DRIVE_COMFY_DIR)} ({m['gib']} GiB)")
        found_models[m['id']] = resolved
    else:
        print(f"  [❌ FALTANDO NO DRIVE ] {m['desc']}")
        print(f"      Destino: models/{m['folder']}/{m['name']} ({m['gib']} GiB)")
        to_download.append(m)

# 5. Download dos modelos ausentes diretamente para o Google Drive
if not to_download:
    print('\n🎉 Todos os modelos já estão baixados e prontos no seu Google Drive!')
else:
    from huggingface_hub import hf_hub_download
    total_download_gib = round(sum(m['gib'] for m in to_download), 2)
    free_drive_gib = round(shutil.disk_usage(str(DRIVE_COMFY_DIR)).free / (1024**3), 2)
    print(f'\n⬇️ Modelos a baixar: {len(to_download)} ({total_download_gib} GiB totais)')
    print(f'💾 Espaço livre no Google Drive: {free_drive_gib} GiB')
    
    if free_drive_gib < total_download_gib + 2:
        raise RuntimeError(f'Espaço insuficiente no Drive: precisa de {total_download_gib + 2} GiB, mas há {free_drive_gib} GiB livres.')
    
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
        
    for m in to_download:
        dest_dir = DRIVE_COMFY_MODELS / m['folder']
        dest_file = dest_dir / m['name']
        print(f'\nBaixando {m["name"]} ({m["gib"]} GiB) direto para {dest_dir.name}...')
        t0 = time.perf_counter()
        try:
            downloaded_path = hf_hub_download(
                repo_id=m['repo'],
                filename=m['hf_filename'],
                local_dir=str(dest_dir),
                token=token if token else False
            )
            # Caso o arquivo venha com subpasta (ex: vae/arquivo.safetensors)
            downloaded_path = Path(downloaded_path)
            if downloaded_path.is_file() and downloaded_path != dest_file:
                shutil.move(str(downloaded_path), str(dest_file))
                # Limpa pasta temporária gerada se vazia
                if downloaded_path.parent != dest_dir and not any(downloaded_path.parent.iterdir()):
                    downloaded_path.parent.rmdir()
            dt = round(time.perf_counter() - t0, 1)
            print(f'✅ Concluído em {dt}s ({round(m["gib"] * 1024 / max(dt, 1), 1)} MiB/s) - Salvo no Drive!')
            found_models[m['id']] = dest_file
        except Exception as e:
            print(f'❌ Erro ao baixar {m["name"]}: {e}')
            raise

# 6. Criar links simbólicos no ComfyUI local apontando para o Google Drive
print('\n🔗 Vinculando modelos do Google Drive ao ComfyUI...')
if COMFY_LOCAL_MODELS.is_dir():
    for m in H3_MODELS:
        source_file = found_models.get(m['id']) or (DRIVE_COMFY_MODELS / m['folder'] / m['name'])
        if source_file.is_file():
            local_folder = COMFY_LOCAL_MODELS / m['folder']
            local_folder.mkdir(parents=True, exist_ok=True)
            local_link = local_folder / m['name']
            if local_link.is_symlink() or local_link.exists():
                local_link.unlink(missing_ok=True)
            local_link.symlink_to(source_file)
            print(f'  🔗 models/{m["folder"]}/{m["name"]} -> Drive')
print('\n✅ Todos os modelos estão vinculados e prontos para geração!')


## Stage 2 -- Conditioning blob, DiT GGUF, Turbo LoRA, the two VAEs

Everything SHA-256 verified. If no conditioning blob exists for your prompt and
geometry, it is encoded here once (one-time ~15 GB text-encoder download) and
cached to Drive.


In [ ]:
# Stage 2 - the conditioning blob, then the DiT GGUF.
#
# The blob comes first: it costs nothing to check, and there is no point pulling
# 6-11 GiB of DiT if there is nothing to sample from. If it is missing this stage
# regenerates it -- download the encoder, encode once, copy the 5 MiB result to
# Drive -- and then stops ComfyUI so the sample stage starts from a process that
# has never loaded the encoder.
models = {"conditioning": {}, "dit": {}}


def persist_blob_to_drive(key):
    """Drive is the only place a 5 MiB blob survives a recycled runtime."""
    if not DRIVE_MOUNTED:
        return {"status": "SKIPPED_NO_DRIVE"}
    DRIVE_CONDITIONING_STORE.mkdir(parents=True, exist_ok=True)
    copied = []
    for suffix in (".pt", ".summary.json"):
        source = CONDITIONING_STORE / f"{key}{suffix}"
        if source.is_file():
            destination = DRIVE_CONDITIONING_STORE / source.name
            shutil.copy2(source, destination)
            copied.append(str(destination))
    # The geometry is baked into the blob, so it is recorded beside it rather than
    # left implicit in a hash-shaped filename.
    manifest = DRIVE_CONDITIONING_STORE / f"{key}.manifest.json"
    manifest.write_text(json.dumps({
        "key": key, "geometry": GEOMETRY, "prompt": PROMPT,
        "model_repo": MODEL_REPO, "model_revision": MODEL_REVISION,
        "text_encoder_sha256": ENCODE_FILES[TEXT_ENCODER_FILE]["sha256"],
        "comfyui_commit": COMFY_COMMIT,
        "expected_latent_shapes": EXPECTED_LATENT_SHAPES,
        "created_at": utc_now()}, indent=2, ensure_ascii=False), encoding="utf-8")
    copied.append(str(manifest))
    return {"status": "OK", "files": copied}


def fetch_verified(relative_name, expected):
    from huggingface_hub import hf_hub_download
    filename = Path(relative_name).name
    subfolder = Path(relative_name).parent.name
    
    alt_subs = [subfolder]
    if subfolder in ("clip", "text_encoders"):
        alt_subs = ["text_encoders", "clip"]
    elif subfolder in ("unet", "diffusion_models"):
        alt_subs = ["diffusion_models", "unet"]
        
    destination = MODEL_STORE / relative_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    
    record = {"expected_bytes": expected["bytes"],
              "expected_gib": round(expected["bytes"] / GIB, 3)}
              
    # 1. Procura se o modelo já existe no Google Drive do usuário ou no disco local
    found = find_model_file(filename, subfolders=alt_subs, min_bytes=expected["bytes"])
    
    if found and found.is_file() and found.stat().st_size == expected["bytes"]:
        print(f"  [✅ REUTILIZANDO DO DRIVE] {filename} ({expected['bytes'] / GIB:.2f} GiB)")
        print(f"      -> Encontrado em: {found}")
        record["source"] = "already_on_drive"
        destination = found
    elif destination.is_file() and destination.stat().st_size == expected["bytes"]:
        record["source"] = "already_on_disk"
        print(f"  reusing {relative_name} ({expected['bytes'] / GIB:.2f} GiB)")
    else:
        print(f"  [⬇️ BAIXANDO] {relative_name} ({expected['bytes'] / GIB:.2f} GiB) direto para o Drive...")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
        started = time.perf_counter()
        try:
            hf_hub_download(repo_id=expected.get("repo", MODEL_REPO),
                            filename=expected.get("hf_filename", relative_name),
                            revision=expected.get("revision", MODEL_REVISION)
                            if expected.get("repo", MODEL_REPO) == MODEL_REPO else None,
                            local_dir=str(MODEL_STORE),
                            token=token if token else False)
        finally:
            del token
        record["source"] = "huggingface"
        record["download_s"] = round(time.perf_counter() - started, 1)
        record["download_mib_per_s"] = round(
            expected["bytes"] / (1024 ** 2) / max(record["download_s"], 1e-6), 1)
        print(f"    {record['download_s']}s ({record['download_mib_per_s']} MiB/s)")
        
        fetched = MODEL_STORE / expected.get("hf_filename", relative_name)
        if fetched != destination and fetched.is_file() and not destination.is_file():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(fetched), str(destination))
            
    if destination.stat().st_size != expected["bytes"]:
        raise RuntimeError(
            f"{relative_name} is {destination.stat().st_size} bytes, expected "
            f"{expected['bytes']}. A truncated safetensors surfaces later as a "
            "misleading shape error, so stop here.")
            
    print(f"  verifying SHA-256 ...")
    record["sha256"] = sha256_file(destination)
    record["sha256_matches"] = record["sha256"] == expected["sha256"]
    if not record["sha256_matches"]:
        raise RuntimeError(f"{relative_name} SHA-256 mismatch; refusing to continue.")
        
    link = COMFY_DIR / "models" / relative_name
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != destination.resolve():
            link.unlink()
            link.symlink_to(destination)
    elif link.exists() and link.resolve() != destination.resolve():
        pass
    elif not link.exists():
        link.symlink_to(destination)
    record["linked_at"] = str(link)
    return record


def regenerate_conditioning():
    """Download the encoder, encode once, save, persist, and shut ComfyUI down."""
    import requests
    print("Regenerating the conditioning blob.")
    print("  19.46 GiB of encoder and VAE for a 5 MiB result. This is why it goes")
    print("  to Drive afterwards.")
    report = {"key": CONDITIONING_KEY, "files": {}}
    for relative_name, expected in ENCODE_FILES.items():
        report["files"][relative_name] = fetch_verified(relative_name, expected)

    system_stats, log_offset = start_comfy(quiet=True)
    encode_object_info = requests.get(f"{BASE_URL}/object_info", timeout=300).json()
    for node in ("CLIPLoader", "VAELoader", "MiniMaxH3ImageToVideo", "H3SaveConditioning"):
        if node not in encode_object_info:
            raise RuntimeError(f"{node} is missing; cannot encode")
    encode_graph = {
        "2": {"class_type": "CLIPLoader", "inputs": {
            "clip_name": Path(TEXT_ENCODER_FILE).name, "type": "minimax",
            "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": Path(VIDEO_VAE_FILE).name}},
        "5": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {
            "clip": ["2", 0], "vae": ["3", 0], "prompt": PROMPT, **GEOMETRY}},
        "40": {"class_type": "H3SaveConditioning", "inputs": {
            "conditioning": ["5", 0], "samples": ["5", 1],
            "store_dir": str(CONDITIONING_STORE), "key": CONDITIONING_KEY}},
    }
    prompt_id = str(uuid.uuid4())
    response = requests.post(f"{BASE_URL}/prompt", timeout=120, json={
        "prompt": encode_graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
    response.raise_for_status()
    if response.json().get("node_errors"):
        raise RuntimeError(f"Node errors: {response.json()['node_errors']}")
    print("  encoding (about 70 s once the encoder has streamed in) ...")
    shown, started = log_offset, time.perf_counter()
    while time.perf_counter() - started < 3600:
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError("ComfyUI exited during the encode (check for an OOM kill)")
        chunk = tail_comfy_log(shown)
        shown += len(chunk.encode("utf-8"))
        for line in chunk.splitlines():
            if line.strip():
                print("    " + line[:150])
        history = requests.get(f"{BASE_URL}/history/{prompt_id}", timeout=60).json()
        if prompt_id in history:
            status = history[prompt_id].get("status", {})
            if status.get("completed") is True or status.get("status_str") == "error":
                if status.get("status_str") != "success":
                    raise RuntimeError(f"Encode failed: {status}")
                break
        time.sleep(3)
    else:
        raise TimeoutError("Encode did not finish")
    report["encode_s"] = round(time.perf_counter() - started, 1)
    print(f"  encoded in {report['encode_s']}s")

    blob = CONDITIONING_STORE / f"{CONDITIONING_KEY}.pt"
    if not blob.is_file():
        raise RuntimeError(f"Encode reported success but produced no blob at {blob}")
    report["blob_bytes"] = blob.stat().st_size
    report["blob_mib"] = round(blob.stat().st_size / (1024 ** 2), 2)
    report["drive"] = persist_blob_to_drive(CONDITIONING_KEY)
    for path in report["drive"].get("files", []):
        print(f"  -> {path}")
    # The sample stage must not inherit the encoder's mapped pages.
    stop_comfy()
    print("  ComfyUI stopped so the sample stage starts clean.")
    return blob, report


blob_path = RESOLVED_BLOB
if blob_path is None:
    if not REGENERATE_CONDITIONING_IF_MISSING:
        models["conditioning"] = {"status": "MISSING", "searched":
                                  PROBE["stages"]["environment"]["blob_searched"]}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(
            "No conditioning blob and regeneration is disabled. Set "
            "REGENERATE_CONDITIONING_IF_MISSING=True, or point CONDITIONING_BLOB at a copy.")
    if not license_gate_open():
        models["conditioning"] = {"status": "MISSING_LICENSE_GATE_LOCKED"}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(
            "The blob has to be regenerated, which downloads MiniMax H3 weights, but the "
            "license gate is locked. Set the five fields in Stage 0.")
    blob_path, regeneration = regenerate_conditioning()
    models["conditioning"] = {"status": "REGENERATED", **regeneration}
else:
    print(f"Using the conditioning blob at {blob_path}")
    if blob_path.parent != CONDITIONING_STORE:
        shutil.copy2(blob_path, CONDITIONING_STORE / blob_path.name)
        summary = blob_path.with_suffix(".summary.json")
        if summary.is_file():
            shutil.copy2(summary, CONDITIONING_STORE / summary.name)
    models["conditioning"] = {
        "status": "REUSED", "key": CONDITIONING_KEY, "source": str(blob_path),
        "blob_bytes": blob_path.stat().st_size,
        "blob_mib": round(blob_path.stat().st_size / (1024 ** 2), 2)}
    # A blob found on Drive may predate this runtime; make sure Drive has it too.
    if DRIVE_MOUNTED and not (DRIVE_CONDITIONING_STORE / f"{CONDITIONING_KEY}.pt").is_file():
        models["conditioning"]["drive"] = persist_blob_to_drive(CONDITIONING_KEY)

print(f"conditioning: {models['conditioning']['status']}  "
      f"{models['conditioning'].get('blob_mib')} MiB  key={CONDITIONING_KEY}")

# --- Turbo LoRA and the two VAEs --------------------------------------------
# These are needed by later stages rather than by sampling: the LoRA by Stage 4,
# the VAEs only by the decode stage, which runs in a different process. They are
# fetched here so one pass over Stage 2 leaves the runtime complete.
models["lora"] = {"choice": TURBO_LORA}
if LORA is None:
    models["lora"]["status"] = "SKIPPED_OFF"
    print("\nTurbo LoRA: off")
elif not RUN_MODEL_DOWNLOAD and not (MODEL_STORE / "loras" / LORA["file"]).is_file():
    models["lora"]["status"] = "DOWNLOAD_DISABLED"
    print("\nTurbo LoRA: missing and RUN_MODEL_DOWNLOAD is False")
else:
    print(f"\nTurbo LoRA {TURBO_LORA} -- {LORA['note']}")
    models["lora"].update(fetch_verified(
        f"loras/{LORA['file']}", {**LORA, "hf_filename": LORA["file"]}))
    models["lora"]["status"] = "READY"
    models["lora"]["strength_note"] = (
        "The file carries no alpha tensor and its metadata says "
        "W_eff = W + lora_B @ lora_A, so alpha = rank and strength 1.0 is the "
        "tuned value. The 0.094 figure belongs to a file with alpha/rank = 1/8.")

models["vae"] = {}
if not DECODE_TO_MP4:
    models["vae"]["status"] = "SKIPPED_DECODE_OFF"
    print("VAEs: skipped (DECODE_TO_MP4 is False)")
elif not RUN_MODEL_DOWNLOAD and not all(
        (MODEL_STORE / spec["file"]).is_file() for spec in VAE_FILES.values()):
    models["vae"]["status"] = "DOWNLOAD_DISABLED"
    print("VAEs: missing and RUN_MODEL_DOWNLOAD is False")
else:
    total = sum(spec["bytes"] for spec in VAE_FILES.values())
    print(f"\nVAEs for the decode stage ({total / GIB:.2f} GiB)")
    for kind, spec in VAE_FILES.items():
        models["vae"][kind] = fetch_verified(spec["file"], spec)
    models["vae"]["status"] = "READY"
    models["vae"]["resident_note"] = (
        f"{total / GIB:.2f} GiB, and the decode process holds no DiT: peak is "
        "max(stage), not sum(stages).")

# --- The DiT ----------------------------------------------------------------
if not license_gate_open():
    models["dit"] = {"status": "SKIPPED_LICENSE_GATE_LOCKED"}
    print("\nDiT SKIPPED: the license gate is locked. Missing:")
    for name, ok in (
        ("I_HAVE_READ_AND_ACCEPTED_MINIMAX_H3_LICENSE",
         I_HAVE_READ_AND_ACCEPTED_MINIMAX_H3_LICENSE),
        ("MINIMAX_LICENSE_BASIS", MINIMAX_LICENSE_BASIS in
         {"applicable_territory", "separate_minimax_authorization"}),
        ("I_CONFIRM_RUNTIME_AND_USE_LOCATIONS_ARE_COVERED",
         I_CONFIRM_RUNTIME_AND_USE_LOCATIONS_ARE_COVERED),
        ("I_CONFIRM_ACCEPTABLE_USE_AND_OUTPUT_RULES",
         I_CONFIRM_ACCEPTABLE_USE_AND_OUTPUT_RULES),
        ("I_ACCEPT_RESPONSIBILITY_FOR_OUTPUTS", I_ACCEPT_RESPONSIBILITY_FOR_OUTPUTS),
    ):
        if not ok:
            print(f"  - {name}")
    print("  These are community quantisations of MiniMax H3; the upstream LICENSE applies.")
else:
    from huggingface_hub import hf_hub_download

    destination = MODEL_STORE / "diffusion_models" / DIT["file"]
    destination.parent.mkdir(parents=True, exist_ok=True)
    record = {"choice": DIT_CHOICE, "repo": DIT["repo"], "file": DIT["file"],
              "expected_bytes": DIT["bytes"], "expected_gib": round(DIT["bytes"] / GIB, 3)}
              
    # 1. Procura se o DiT já existe no Google Drive do usuário ou em subpastas diffusion_models/unet
    found_dit = find_model_file(DIT["file"], subfolders=["diffusion_models", "unet"], min_bytes=DIT["bytes"])
    present = destination.is_file() and destination.stat().st_size == DIT["bytes"]
    free_gib = shutil.disk_usage("/content").free / GIB

    if found_dit and found_dit.is_file() and found_dit.stat().st_size == DIT["bytes"]:
        print(f"\n[✅ REUTILIZANDO DiT DO DRIVE] {DIT['file']} ({DIT['bytes'] / GIB:.2f} GiB)")
        print(f"    -> Encontrado em: {found_dit}")
        destination = found_dit
        record["source"] = "already_on_drive"
    elif present:
        print(f"\nReusing {DIT['file']} ({DIT['bytes'] / GIB:.2f} GiB)")
        record["source"] = "already_on_disk"
    elif not RUN_MODEL_DOWNLOAD:
        models["dit"] = {"status": "DOWNLOAD_DISABLED", **record}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(f"{DIT['file']} is missing and RUN_MODEL_DOWNLOAD is False.")
    elif free_gib < DIT["bytes"] / GIB + 5:
        models["dit"] = {"status": "INSUFFICIENT_DISK", "free_gib": round(free_gib, 2), **record}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(f"Only {free_gib:.1f} GiB free; need {DIT['bytes'] / GIB + 5:.1f} GiB.")
    else:
        print(f"\n[⬇️ BAIXANDO DiT] {DIT['file']} ({DIT['bytes'] / GIB:.2f} GiB) direto para o Drive...")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
        started = time.perf_counter()
        try:
            fetched = hf_hub_download(repo_id=DIT["repo"], filename=DIT["file"],
                                      local_dir=str(MODEL_STORE / "_hf"),
                                      token=token if token else False)
        finally:
            del token
        shutil.move(fetched, destination)
        record["source"] = "huggingface"
        record["download_s"] = round(time.perf_counter() - started, 1)
        record["download_mib_per_s"] = round(
            DIT["bytes"] / (1024 ** 2) / max(record["download_s"], 1e-6), 1)
        print(f"  {record['download_s']}s ({record['download_mib_per_s']} MiB/s)")

    actual = destination.stat().st_size
    record["actual_bytes"] = actual
    if actual != DIT["bytes"]:
        models["dit"] = {"status": "WRONG_SIZE", **record}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(
            f"{DIT['file']} is {actual} bytes, expected {DIT['bytes']}. A truncated model "
            "surfaces later as a misleading shape error, so stop here.")
    print(f"  verifying SHA-256 ({DIT['bytes'] / GIB:.2f} GiB) ...")
    started = time.perf_counter()
    record["sha256"] = sha256_file(destination)
    record["sha256_matches"] = record["sha256"] == DIT["sha256"]
    record["verify_s"] = round(time.perf_counter() - started, 1)
    if not record["sha256_matches"]:
        models["dit"] = {"status": "SHA256_MISMATCH", **record}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(f"{DIT['file']} SHA-256 mismatch; refusing to continue.")
    print(f"  verified in {record['verify_s']}s")

    # What the file actually declares, read from its own header. The expected value
    # is recorded in DIT_CHOICES; a mismatch means the repo changed under us.
    def gguf_architecture(path):
        try:
            from gguf import GGUFReader
            reader = GGUFReader(str(path))
        except Exception as exc:
            return {"status": f"reader_unavailable:{type(exc).__name__}"}
        try:
            field = reader.fields.get("general.architecture")
            if field is None:
                return {"status": "OK", "architecture": None}
            return {"status": "OK",
                    "architecture": bytes(field.parts[field.data[0]]).decode("utf-8", "replace")}
        except Exception as exc:
            return {"status": f"parse_failed:{type(exc).__name__}"}

    header = gguf_architecture(destination)
    record["gguf_header"] = header
    record["gguf_arch_expected"] = DIT.get("gguf_arch")
    print(f"  general.architecture: {header.get('architecture')!r} "
          f"(expected {DIT.get('gguf_arch')!r})")
    if header.get("status") == "OK" and header.get("architecture") is None:
        models["dit"] = {"status": "NO_ARCHITECTURE_IN_HEADER", **record}
        PROBE["stages"]["models"] = models
        save()
        raise RuntimeError(
            f"{DIT['file']} declares no general.architecture. ComfyUI-GGUF would fall "
            "into its compatibility branch and fail with 'Unknown model architecture!'. "
            "Pick a repo whose files declare one.")
    if header.get("status") == "OK" and header.get("architecture") != DIT.get("gguf_arch"):
        print(f"  NOTE: header says {header.get('architecture')!r}, DIT_CHOICES expected "
              f"{DIT.get('gguf_arch')!r}. Recorded, not enforced.")

    link = COMFY_DIR / "models" / "diffusion_models" / DIT["file"]
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != destination.resolve():
            link.unlink()
            link.symlink_to(destination)
    elif link.exists():
        raise RuntimeError(f"Refusing to replace a non-symlink model path: {link}")
    else:
        link.symlink_to(destination)
    record["linked_at"] = str(link)
    record["status"] = "READY"
    record["resident_note"] = (
        f"{DIT['bytes'] / GIB:.2f} GiB of DiT. The sampling graph has no text encoder and "
        "no VAE: the encoder's output arrives as a 5 MiB blob and decoding is a separate "
        "stage.")
    models["dit"] = record

PROBE["stages"]["models"] = models
save()
print()
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "files"}
                  for k, v in models.items()}, indent=2, ensure_ascii=False)[:2500])


## Stage 3 -- Launch ComfyUI, verify nodes, dtype, and patches

The requested dtype is checked against the server's actual argv; a mismatch stops
the run here instead of wasting a 30-minute sample.


In [ ]:
# Stage 3 - start ComfyUI and confirm the GGUF loader accepts minimax_h3.
# ComfyUI process control lives in Stage 1 so that Stage 2 can use it too.
import requests

system_stats, startup_log_offset = start_comfy()
object_info = requests.get(f"{BASE_URL}/object_info", timeout=300).json()

required = {"UnetLoaderGGUF", "H3LoadConditioning", "H3LatentProbe",
            "RandomNoise", "BasicScheduler", "KSamplerSelect", "BasicGuider",
            "SamplerCustomAdvanced"}
required |= {"H3SaveLatent"}
if PERF_PROBE:
    required |= {"H3PerfProbe"}
if LORA:
    required |= {"MiniMaxH3TurboLoRA", "MiniMaxH3TurboSampler"}
if DECODE_TO_MP4:
    # Checked here rather than in the decode stage: finding out that SaveVideo is
    # absent after a 14-minute sample would be a waste of the sample.
    required |= {"H3LoadLatent", "VAELoader", "VAEDecode", "VAEDecodeAudio",
                 "CreateVideo", "SaveVideo"}
missing = sorted(required - set(object_info))
if missing:
    raise RuntimeError(f"Missing required nodes: {missing}")


def dropdown(node_type, input_name):
    inputs = object_info[node_type].get("input", {})
    spec = (inputs.get("required", {}) | inputs.get("optional", {})).get(input_name)
    return spec[0] if spec and isinstance(spec[0], list) else []


gguf_options = dropdown("UnetLoaderGGUF", "unet_name")
DIT_VISIBLE = DIT["file"] in gguf_options
lora_options = dropdown("MiniMaxH3TurboLoRA", "lora_name") if LORA else []
LORA_VISIBLE = (LORA["file"] in lora_options) if LORA else None
vae_options = dropdown("VAELoader", "vae_name") if DECODE_TO_MP4 else []
VAE_VISIBLE = ({kind: Path(spec["file"]).name in vae_options
                for kind, spec in VAE_FILES.items()} if DECODE_TO_MP4 else {})
sampler_options = dropdown("KSamplerSelect", "sampler_name")
scheduler_options = dropdown("BasicScheduler", "scheduler")
for name, value, options in (("sampler", SAMPLER, sampler_options),
                             ("scheduler", SCHEDULER, scheduler_options)):
    if options and value not in options:
        raise RuntimeError(f"{name} {value!r} is not offered; options: {options}")

startup_log = tail_comfy_log(startup_log_offset)
(WORK / "comfy_startup_log.txt").write_text(startup_log, encoding="utf-8")

BACKEND_RE = re.compile(
    r"Found comfy_kitchen backend (\w+): \{'available': (True|False), 'disabled': (True|False)")
backends = {name: {"available": a == "True", "disabled": d == "True",
                   "usable": a == "True" and d == "False"}
            for name, a, d in BACKEND_RE.findall(startup_log)}

PROBE["stages"]["comfyui"] = {
    "flags": COMFY_FLAG_DECISION,
    "argv": system_stats.get("system", {}).get("argv"),
    "comfyui_version": system_stats.get("system", {}).get("comfyui_version"),
    "comfy_kitchen_backends": backends,
    "usable_backends": sorted(n for n, s in backends.items() if s["usable"]),
    "gguf_unet_options": gguf_options,
    "dit_visible_to_gguf_loader": DIT_VISIBLE,
    "lora_visible_to_turbo_node": LORA_VISIBLE,
    "turbo_lora_inputs": sorted(
        (object_info.get("MiniMaxH3TurboLoRA", {}).get("input", {}).get("required") or {}).keys())
    if LORA else None,
    "vae_visible": VAE_VISIBLE,
    "unet_loader_gguf_inputs": sorted(
        (object_info["UnetLoaderGGUF"].get("input", {}).get("required") or {}).keys()),
    "import_failures": [l for l in startup_log.splitlines() if "IMPORT FAILED" in l],
    "pinned_memory_lines": [l.strip() for l in startup_log.splitlines()
                            if "pinned memory" in l or "DynamicVRAM" in l],
}
save()

# Three runs were spoiled by a patched start_comfy disagreeing with the config, so
# what ComfyUI actually received is checked against what was requested.
applied_flags = PROBE["stages"]["comfyui"]["flags"]["applied"]
dtype_flags_seen = [f for f in applied_flags if f in set(UNET_DTYPE_FLAGS.values()) - {None}]
expected_dtype_flags = [UNET_DTYPE_FLAG] if UNET_DTYPE_FLAG else []
if dtype_flags_seen != expected_dtype_flags:
    raise RuntimeError(
        f"ComfyUI was started with dtype flags {dtype_flags_seen}, but UNET_DTYPE="
        f"{UNET_DTYPE!r} asks for {expected_dtype_flags}. Any measurement taken now "
        "would describe a configuration nobody chose. Re-run Stage 0 and Stage 1 to "
        "clear stale state.")
# The patch logs from the node's import, so its absence from the startup log means
# it did not load -- and a measurement taken then would not be testing it.
rmsnorm_lines = [l.strip() for l in startup_log.splitlines() if "H3_FP32_RMSNORM" in l]
if RMSNORM_FP32 and not rmsnorm_lines:
    raise RuntimeError(
        "RMSNORM_FP32 is on but the node never logged H3_FP32_RMSNORM. The patch did "
        "not reach the running process; re-run Stage 1 so the control node is rewritten.")
if not RMSNORM_FP32 and rmsnorm_lines:
    raise RuntimeError(
        f"RMSNORM_FP32 is off but the running process applied it: {rmsnorm_lines}. "
        "Re-run Stage 1 to rewrite the control node.")
PROBE["stages"]["comfyui"]["rmsnorm_fp32"] = RMSNORM_FP32
PROBE["stages"]["comfyui"]["rmsnorm_fp32_log"] = rmsnorm_lines
PROBE["stages"]["comfyui"]["unet_dtype"] = UNET_DTYPE
PROBE["stages"]["comfyui"]["unet_dtype_flags_verified"] = dtype_flags_seen
save()

print()
print("=" * 72)
print(f"unet dtype              : {UNET_DTYPE} -> {dtype_flags_seen or '(default)'}  VERIFIED")
print(f"fp32 RMSNorm            : {RMSNORM_FP32}  {rmsnorm_lines or ''}")
print(f"UnetLoaderGGUF inputs   : {PROBE['stages']['comfyui']['unet_loader_gguf_inputs']}")
print(f"DiT visible to loader   : {DIT_VISIBLE}  ({DIT['file']})")
if LORA:
    print(f"LoRA visible to Turbo   : {LORA_VISIBLE}  ({LORA['file']})")
    print(f"  MiniMaxH3TurboLoRA inputs: "
          f"{PROBE['stages']['comfyui']['turbo_lora_inputs']}")
if DECODE_TO_MP4:
    print(f"VAEs visible            : {VAE_VISIBLE}")
print(f"comfy_kitchen usable    : {PROBE['stages']['comfyui']['usable_backends']}")
for line in PROBE["stages"]["comfyui"]["pinned_memory_lines"]:
    print("    " + line[:150])
if PROBE["stages"]["comfyui"]["import_failures"]:
    print(f"IMPORT FAILURES: {PROBE['stages']['comfyui']['import_failures']}")
print("=" * 72)
if not DIT_VISIBLE:
    print("The DiT is not visible to UnetLoaderGGUF. Stage 4 cannot run.")
    print(f"  options seen: {gguf_options[:10]}")
print()
print("Note: the architecture patch is not proven until a GGUF is actually read.")
print("Stage 4 reads one and reports arch_error_seen if the patch did not hold.")
print("(Nothing above is an error. This is the only place that mentions rejection.)")


## Stage 4 -- Sampling (Turbo LoRA + cliff guard + latent saved to disk)

The graph holds the DiT and nothing else -- no text encoder, no VAE. A guard
projects the finish time from live progress and cancels over-budget runs early.


In [ ]:
# Stage 4 - sample, with a guard that projects the finish time and gives up early.
#
# The graph is seven nodes and contains neither a CLIPLoader nor a VAELoader: the
# text encoder's output arrives as a 5 MiB blob and decoding is a separate stage.
# Resident weights are the DiT and nothing else.
# Stage 3 defines DIT_VISIBLE and object_info, and it is also what starts ComfyUI.
# Reaching here without it means Stage 3 was skipped; say so rather than raising a
# bare NameError on a variable the reader has no reason to know about.
for _needed in ("DIT_VISIBLE", "object_info"):
    if _needed not in globals():
        raise RuntimeError(
            f"{_needed} is not defined, so Stage 3 has not run. Stage 3 starts ComfyUI "
            "and checks that the DiT is visible to UnetLoaderGGUF. Run it, then come "
            "back here.")

if not DIT_VISIBLE:
    PROBE["stages"]["sample"] = {"status": "SKIPPED_DIT_NOT_VISIBLE"}
    save()
    raise RuntimeError("The DiT is not visible to UnetLoaderGGUF; run Stage 2 first.")

label = (f"sample_{DIT_CHOICE}_{UNET_DTYPE}_{TURBO_LORA}_{STEPS}step_gear{GEAR}_"
         f"{GEOMETRY['width']}x{GEOMETRY['height']}_{GEOMETRY['length']}f")
filename_prefix = f"h3_sample/{RUN_ID}/{label}"
# The decode stage runs in a different process and finds the latent by this key.
LATENT_KEY = f"{RUN_ID}_{label}"


def build_sample_graph():
    # Both control nodes are pass-throughs on MODEL. Chain whichever are enabled
    # between the loader and its consumers; model_src ends up pointing at the last.
    # The fp32 patch goes first so the tracer observes the patched behaviour.
    model_src = ["1", 0]
    extra = {}
    if LORA:
        # MiniMaxH3TurboLoRA clones the patcher; the fp32 and trace patches act on
        # the shared nn.Module underneath, so the order between them is free.
        extra["19"] = {"class_type": "MiniMaxH3TurboLoRA", "inputs": {
            "model": model_src, "lora_name": LORA["file"],
            "strength": LORA_STRENGTH, "low_vram": LORA_LOW_VRAM}}
        model_src = ["19", 0]
    if FP32_TEXT_BRANCH:
        extra["17"] = {"class_type": "H3Fp32TextBranch", "inputs": {"model": model_src}}
        model_src = ["17", 0]
    if NAN_TRACE:
        extra["16"] = {"class_type": "H3NanTrace",
                       "inputs": {"model": model_src, "trail_len": 16}}
        model_src = ["16", 0]
    if PERF_PROBE:
        # Last in the chain so it instruments the module the run actually uses.
        extra["20"] = {"class_type": "H3PerfProbe", "inputs": {"model": model_src}}
        model_src = ["20", 0]
    graph = {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": DIT["file"]}},
        "41": {"class_type": "H3LoadConditioning", "inputs": {
            "store_dir": str(CONDITIONING_STORE), "key": CONDITIONING_KEY}},
        "6": {"class_type": "RandomNoise", "inputs": {"noise_seed": SEED}},
        "7": {"class_type": "BasicScheduler", "inputs": {
            "model": model_src, "scheduler": SCHEDULER, "steps": STEPS, "denoise": 1.0}},
        # The Turbo sampler steps video and audio on their own flow schedules; a
        # stock single-schedule sampler over-steps the audio at 4 steps.
        "8": ({"class_type": "MiniMaxH3TurboSampler", "inputs": {}} if LORA
              else {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER}}),
        "9": {"class_type": "BasicGuider", "inputs": {
            "model": model_src, "conditioning": ["41", 0]}},
        "10": {"class_type": "SamplerCustomAdvanced", "inputs": {
            "noise": ["6", 0], "guider": ["9", 0], "sampler": ["8", 0],
            "sigmas": ["7", 0], "latent_image": ["41", 1]}},
        # Save before probing: the probe raises on a non-finite latent, and a
        # failed run is exactly when the artefact is worth having.
        "18": {"class_type": "H3SaveLatent", "inputs": {
            "samples": ["10", 0], "store_dir": str(LATENT_STORE), "key": LATENT_KEY}},
        "15": {"class_type": "H3LatentProbe", "inputs": {
            "samples": ["18", 0], "filename_prefix": filename_prefix}},
    }
    graph.update(extra)
    return graph


def validate(graph):
    errors = []
    serialized = json.dumps(graph)
    for forbidden, why in (("CLIPLoader", "the 14.61 GiB text encoder must not load here"),
                           ("VAELoader", "decoding is a separate stage"),
                           ("MiniMaxH3ImageToVideo", "that would re-run the encoder")):
        if forbidden in serialized:
            errors.append(f"{forbidden} present: {why}")
    for node_id, spec in graph.items():
        info = object_info.get(spec["class_type"])
        if info is None:
            errors.append(f"{node_id}: unknown class {spec['class_type']}")
            continue
        schema = info.get("input", {})
        allowed = set(schema.get("required", {})) | set(schema.get("optional", {})) \
            | set(schema.get("hidden", {}))
        supplied = set(spec["inputs"])
        if set(schema.get("required", {})) - supplied:
            errors.append(f"{node_id}/{spec['class_type']} missing "
                          f"{sorted(set(schema.get('required', {})) - supplied)}")
        if supplied - allowed:
            errors.append(f"{node_id}/{spec['class_type']} unknown {sorted(supplied - allowed)}")
    if errors:
        raise RuntimeError("Sample graph validation failed:\n" + "\n".join(errors))


graph = build_sample_graph()
validate(graph)
(WORK / "sample_graph.json").write_text(json.dumps(graph, indent=2), encoding="utf-8")
print(f"Sample graph validated: {sorted(n['class_type'] for n in graph.values())}")
print(f"  no CLIPLoader, no VAELoader. Resident weights: the DiT alone "
      f"({DIT['bytes'] / GIB:.2f} GiB)")

ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")
TQDM_RE = re.compile(
    r"(\d+)%\|[^|\n]*\|\s*(\d+)/(\d+)\s*\[([0-9:]+)<([0-9:]+|\?)"
    r"(?:,\s*([0-9.]+)(s/it|it/s))?")


def parse_clock(value):
    total = 0.0
    for part in value.split(":"):
        total = total * 60 + float(part)
    return total


def latest_progress(text):
    """Latest SamplerCustomAdvanced progress. ComfyUI's own ETA is not used."""
    matches = list(TQDM_RE.finditer(ANSI_RE.sub("", text).replace("\r", "\n")))
    if not matches:
        return None
    percent, done, total, elapsed, _remaining, rate, unit = matches[-1].groups()
    done, elapsed_s = int(done), parse_clock(elapsed)
    return {"percent": int(percent), "steps_completed": done, "steps_total": int(total),
            "elapsed_in_loop_s": elapsed_s,
            "observed_s_per_step": (elapsed_s / done) if done else None,
            "reported_s_per_step": (float(rate) if unit == "s/it"
                                    else (1.0 / float(rate) if rate and float(rate) else None))
            if rate else None}


# --- The guard, as a pure function ------------------------------------------
# Previous versions inlined this in the polling loop and twice got it wrong: once
# by mixing time.time() with time.perf_counter(), once by a stray indent that put
# the projection logic inside the "no progress yet" branch. Pulling it out makes
# it testable without a GPU, which is how those would have been caught.
GUARD_HEARTBEAT_S = 30


def new_guard_runtime(started):
    return {"started": started, "sampler_started_at": None, "windows_used": 0,
            "last_heartbeat": started, "settled": False, "previous_window": None}


def guard_decide(progress, now, runtime, guard_state):
    """Decide whether to keep waiting or cancel. No I/O, no globals.

    progress is latest_progress()'s dict or None. Returns (decision, runtime)
    where decision has stop/outcome/messages.
    """
    runtime = dict(runtime)
    messages = []
    waited = now - runtime["started"]
    if waited < 0 or waited > 86400:
        raise RuntimeError(
            f"Guard clock is inconsistent: waited={waited}. Every reading must come "
            "from time.perf_counter().")

    if progress is None:
        if now - runtime["last_heartbeat"] >= GUARD_HEARTBEAT_S:
            runtime["last_heartbeat"] = now
            messages.append(f"[guard] {waited:.0f}s: loading, no sampler progress yet "
                            f"(timeout {SAMPLER_START_TIMEOUT_S}s)")
        if waited > SAMPLER_START_TIMEOUT_S:
            guard_state["seconds_waited_for_sampler"] = round(waited, 1)
            return ({"stop": True, "outcome": "SAMPLER_NEVER_STARTED",
                     "messages": messages}, runtime)
        return ({"stop": False, "outcome": None, "messages": messages}, runtime)

    if runtime["sampler_started_at"] is None:
        runtime["sampler_started_at"] = now - progress["elapsed_in_loop_s"]
        guard_state["sampler_started"] = True
        guard_state["seconds_to_sampler_start"] = round(
            runtime["sampler_started_at"] - runtime["started"], 1)
        messages.append(f"[guard] sampler started after "
                        f"{guard_state['seconds_to_sampler_start']}s")

    if runtime["settled"]:
        return ({"stop": False, "outcome": "WITHIN_BUDGET", "messages": messages}, runtime)

    observed = now - runtime["sampler_started_at"]

    # The no-step deadline is checked on every poll, not only when a window
    # boundary happens to fall. Tying it to the windows made the cancel land up to
    # a whole window late, which is the opposite of what a guard is for.
    #
    # It comes from the budget rather than from a window count. The old rule was
    # "two 120 s windows with no step completed is a cliff", which silently
    # assumed a step is short compared to the window. At 736x416 a step takes
    # about 235 s: the bypass run cleared 240 s by five seconds and the merge run
    # did not, and a healthy run was cancelled as a cliff.
    #
    # A step that has not finished in twice its budgeted share cannot let the run
    # finish inside the budget. That scales with STEPS and with the budget instead
    # of pretending to know how long a step should take. The window floor is kept
    # so that a tiny budget cannot cancel before there is anything to see.
    if progress["steps_completed"] <= 0:
        no_step_deadline = max(CLIFF_PROBE_WINDOW_S * CLIFF_PROBE_MAX_WINDOWS,
                               2.0 * CLIFF_PROJECTED_BUDGET_S / max(STEPS, 1))
        guard_state["no_step_deadline_s"] = round(no_step_deadline, 1)
        if observed >= no_step_deadline:
            guard_state["observation_s"] = round(observed, 1)
            guard_state["steps_completed_at_decision"] = 0
            messages.append(f"[guard] {observed:.0f}s observed and not one step has "
                            f"finished (deadline {no_step_deadline:.0f}s)")
            return ({"stop": True, "outcome": "CLIFF_NO_STEP_COMPLETED",
                     "messages": messages}, runtime)

    if observed < CLIFF_PROBE_WINDOW_S * (runtime["windows_used"] + 1):
        return ({"stop": False, "outcome": None, "messages": messages}, runtime)

    runtime["windows_used"] += 1
    # elapsed/done counts the step currently in flight, so it overestimates the
    # rate early on: 1 step done at 120 s reads as 120 s/step even if the true
    # rate is 85. The bias shrinks as steps accumulate and is deliberately kept,
    # because it errs towards cancelling. It does not risk misreading the thing
    # being measured: the cliff is orders of magnitude, not tens of percent.
    # The marginal rate between windows is recorded alongside for reading later.
    marginal = None
    previous = runtime["previous_window"]
    if previous and progress["steps_completed"] > previous["steps_completed"]:
        marginal = ((progress["elapsed_in_loop_s"] - previous["elapsed_in_loop_s"])
                    / (progress["steps_completed"] - previous["steps_completed"]))
    runtime["previous_window"] = {
        "elapsed_in_loop_s": progress["elapsed_in_loop_s"],
        "steps_completed": progress["steps_completed"]}
    guard_state.update({"windows_used": runtime["windows_used"],
                        "observation_s": round(observed, 1),
                        "steps_completed_at_decision": progress["steps_completed"],
                        "observed_s_per_step": progress["observed_s_per_step"],
                        "marginal_s_per_step": round(marginal, 2) if marginal else None})
    if progress["steps_completed"] <= 0:
        messages.append(f"[guard] {observed:.0f}s observed, still 0 steps "
                        f"(cancel at {guard_state.get('no_step_deadline_s')}s)")
        return ({"stop": False, "outcome": None, "messages": messages}, runtime)

    projected = progress["observed_s_per_step"] * STEPS
    guard_state["projected_denoise_total_s"] = round(projected, 1)
    messages.append(f"[guard] {progress['steps_completed']}/{STEPS} steps in "
                    f"{observed:.0f}s -> {progress['observed_s_per_step']:.1f} s/step, "
                    f"projected {projected / 60:.1f} min")
    if projected > CLIFF_PROJECTED_BUDGET_S:
        return ({"stop": True, "outcome": "CLIFF_PROJECTED_OVER_BUDGET",
                 "messages": messages}, runtime)
    runtime["settled"] = True
    guard_state["outcome"] = "WITHIN_BUDGET"
    return ({"stop": False, "outcome": "WITHIN_BUDGET", "messages": messages}, runtime)


samples_log, stop_event = [], threading.Event()


def watch():
    with (WORK / f"samples_{label}.jsonl").open("w", encoding="utf-8") as handle:
        while not stop_event.is_set():
            virtual = psutil.virtual_memory()
            record = {"t": round(time.time(), 2),
                      "host_available_gib": round(virtual.available / GIB, 3)}
            try:
                root = psutil.Process(COMFY_PROCESS.pid)
                rss = root.memory_info().rss
                for child in root.children(recursive=True):
                    try:
                        rss += child.memory_info().rss
                    except psutil.Error:
                        pass
                record["comfy_rss_gib"] = round(rss / GIB, 3)
            except Exception:
                pass
            try:
                line = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used,memory.total",
                     "--format=csv,noheader,nounits"], text=True, timeout=5,
                    env=scrubbed_child_environment()).strip().splitlines()[0]
                used, total = [float(x) for x in line.split(",")]
                record["vram_used_mib"], record["vram_headroom_mib"] = used, total - used
            except Exception:
                pass
            handle.write(json.dumps(record) + "\n")
            handle.flush()
            samples_log.append(record)
            stop_event.wait(0.5)


def cancel(prompt_id):
    outcome = {"cancel_requested": False, "drained": False}
    try:
        response = requests.post(f"{BASE_URL}/api/jobs/{prompt_id}/cancel", timeout=30)
        response.raise_for_status()
        outcome["cancel_requested"] = bool(response.json().get("cancelled"))
    except Exception as exc:
        outcome["cancel_error"] = type(exc).__name__
    deadline = time.time() + 120
    while time.time() < deadline:
        try:
            queue = requests.get(f"{BASE_URL}/queue", timeout=30).json()
            active = {item[1] for item in queue.get("queue_running", [])
                      + queue.get("queue_pending", []) if isinstance(item, list) and len(item) > 1}
            if prompt_id not in active:
                outcome["drained"] = True
                return outcome
        except Exception:
            pass
        time.sleep(2)
    return outcome


if COMFY_LOG_HANDLE is not None:
    COMFY_LOG_HANDLE.flush()
log_offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
prompt_id = str(uuid.uuid4())
sample = {"label": label, "prompt_id": prompt_id, "dit_choice": DIT_CHOICE,
          "dit_gib": round(DIT["bytes"] / GIB, 3), "geometry": GEOMETRY, "steps": STEPS,
          "guard": {"window_s": CLIFF_PROBE_WINDOW_S, "max_windows": CLIFF_PROBE_MAX_WINDOWS,
                    "projected_budget_s": CLIFF_PROJECTED_BUDGET_S}}
watcher = threading.Thread(target=watch, daemon=True)
watcher.start()
started = time.perf_counter()
record = None
guard_state = {"outcome": None, "sampler_started": False}

try:
    response = requests.post(f"{BASE_URL}/prompt", timeout=120, json={
        "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
    response.raise_for_status()
    queued = response.json()
    if queued.get("node_errors"):
        raise RuntimeError(f"Node errors: {queued['node_errors']}")
    print(f"submitted; {STEPS} steps at {GEOMETRY['width']}x{GEOMETRY['height']} / "
          f"{GEOMETRY['length']} frames")

    shown = log_offset
    guard_runtime = new_guard_runtime(started)
    # Every clock below is perf_counter. An earlier version compared a
    # time.time() reading against a perf_counter origin, which made the
    # difference about 1.7e9 seconds and fired the start timeout immediately.
    deadline = started + SAMPLER_START_TIMEOUT_S + CLIFF_PROJECTED_BUDGET_S + 1800
    while time.perf_counter() < deadline:
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError("ComfyUI exited during sampling (check for an OOM kill)")
        if COMFY_LOG.exists():
            with COMFY_LOG.open("rb") as reader:
                reader.seek(shown)
                chunk = reader.read().decode("utf-8", errors="replace")
                shown += len(chunk.encode("utf-8"))
            for line in chunk.splitlines():
                if line.strip():
                    print("    " + line[:160])
        try:
            history = requests.get(f"{BASE_URL}/history/{prompt_id}", timeout=60).json()
            if prompt_id in history:
                status = history[prompt_id].get("status", {})
                if status.get("completed") is True or status.get("status_str") == "error":
                    record = history[prompt_id]
                    break
        except Exception:
            pass

        with COMFY_LOG.open("rb") as reader:
            reader.seek(log_offset)
            slice_text = reader.read().decode("utf-8", errors="replace")
        progress = latest_progress(slice_text)
        now = time.perf_counter()
        decision, guard_runtime = guard_decide(progress, now, guard_runtime, guard_state)
        for message in decision["messages"]:
            print("    " + message)
        if decision["stop"]:
            guard_state["outcome"] = decision["outcome"]
            guard_state["cancellation"] = cancel(prompt_id)
            break
        time.sleep(3)
    else:
        raise TimeoutError("Sampling did not finish within the overall deadline")

    if guard_state["outcome"] in ("CLIFF_NO_STEP_COMPLETED", "CLIFF_PROJECTED_OVER_BUDGET",
                                  "SAMPLER_NEVER_STARTED"):
        sample["status"] = "CLIFF"
        sample["failure_class"] = guard_state["outcome"]
    elif record is None:
        sample["status"] = "NO_HISTORY"
    else:
        status = record.get("status", {})
        sample["status_str"] = status.get("status_str")
        sample["status"] = "SUCCESS" if status.get("status_str") == "success" else "FAILED"
        if sample["status"] != "SUCCESS":
            sample["history_status"] = status
except Exception as exc:
    sample["status"] = "FAILED"
    sample["error"] = f"{type(exc).__name__}: {exc}"
finally:
    stop_event.set()
    watcher.join(timeout=5)
    sample["elapsed_s"] = round(time.perf_counter() - started, 1)
    sample["guard_state"] = guard_state
    sample_log = tail_comfy_log(log_offset)
    (WORK / f"sample_log_{label}.txt").write_text(sample_log, encoding="utf-8")
    sample["final_progress"] = latest_progress(sample_log)
    sample["arch_error_seen"] = "Unexpected architecture type" in sample_log
    # H3LatentProbe writes the report before raising, so the trace survives the
    # failure it exists to explain -- but the log is the more reliable carrier.
    trace_lines = [l for l in sample_log.splitlines() if "H3_NAN_TRACE " in l]
    if trace_lines:
        try:
            sample["nan_trace"] = json.loads(
                trace_lines[-1].split("H3_NAN_TRACE ", 1)[1])
        except ValueError as exc:
            sample["nan_trace"] = {"parse_error": str(exc), "line": trace_lines[-1][:400]}
    armed = [l for l in sample_log.splitlines() if "H3_NAN_TRACE_ARM" in l]
    if armed:
        sample["nan_trace_arm"] = [l.strip() for l in armed]
    if NAN_TRACE and not armed:
        sample["nan_trace_arm"] = "NOT ARMED: H3NanTrace never ran"
    # The perf probe logs one H3_PERF_STEP line per model forward; the report
    # line only appears if the run reached H3LatentProbe. A cancelled cliff run
    # still leaves its step lines, which is the point of logging them one by one.
    perf_steps = []
    for line in sample_log.splitlines():
        if "H3_PERF_STEP " in line:
            try:
                perf_steps.append(json.loads(line.split("H3_PERF_STEP ", 1)[1]))
            except ValueError:
                pass
    if perf_steps:
        sample["perf_steps"] = perf_steps
    perf_report_lines = [l for l in sample_log.splitlines()
                         if "H3_PERF_REPORT " in l]
    if perf_report_lines:
        try:
            sample["perf_report"] = json.loads(
                perf_report_lines[-1].split("H3_PERF_REPORT ", 1)[1])
        except ValueError as exc:
            sample["perf_report"] = {"parse_error": str(exc)}
    perf_armed = [l.strip() for l in sample_log.splitlines() if "H3_PERF_ARM" in l]
    if perf_armed:
        sample["perf_probe_arm"] = perf_armed
    if PERF_PROBE and not perf_armed:
        sample["perf_probe_arm"] = "NOT ARMED: H3PerfProbe never ran"
    # The Turbo node prints how many modules it actually patched. Zero means the
    # LoRA was loaded and ignored -- the failure mode that otherwise only shows up
    # as "the video looks like the base model".
    turbo_lines = [l.strip() for l in sample_log.splitlines()
                   if "[MiniMaxH3TurboLoRA]" in l]
    if turbo_lines:
        sample["turbo_lora_log"] = turbo_lines
    if LORA and not turbo_lines:
        sample["turbo_lora_log"] = "NOT APPLIED: MiniMaxH3TurboLoRA never logged"
    lora_key_warnings = [l.strip() for l in sample_log.splitlines()
                         if "lora key not loaded" in l]
    if lora_key_warnings:
        sample["lora_keys_not_loaded"] = len(lora_key_warnings)
        sample["lora_keys_not_loaded_examples"] = lora_key_warnings[:5]
    saved = [l for l in sample_log.splitlines() if "H3_SAVE_LATENT " in l]
    if saved:
        try:
            sample["saved_latent"] = json.loads(saved[-1].split("H3_SAVE_LATENT ", 1)[1])
        except ValueError as exc:
            sample["saved_latent"] = {"parse_error": str(exc)}
    fp32_text = [l.strip() for l in sample_log.splitlines()
                 if "H3_FP32_TEXT_BRANCH" in l]
    if fp32_text:
        sample["fp32_text_branch_log"] = fp32_text
    if FP32_TEXT_BRANCH and not fp32_text:
        sample["fp32_text_branch_log"] = "NOT APPLIED: H3Fp32TextBranch never ran"
    sample["oom_seen"] = bool(re.search(r"CUDA out of memory|torch\.OutOfMemoryError",
                                        sample_log, re.I))
    if record is not None:
        (WORK / f"history_{label}.json").write_text(json.dumps(record, indent=2),
                                                    encoding="utf-8")
    if samples_log:
        rss = [s["comfy_rss_gib"] for s in samples_log if "comfy_rss_gib" in s]
        sample.update({
            "sample_count": len(samples_log),
            "peak_comfy_rss_gib": max(rss) if rss else None,
            "min_host_available_gib": min(s["host_available_gib"] for s in samples_log),
            "peak_vram_used_mib": max((s["vram_used_mib"] for s in samples_log
                                       if "vram_used_mib" in s), default=None),
            "min_vram_headroom_mib": min((s["vram_headroom_mib"] for s in samples_log
                                          if "vram_headroom_mib" in s), default=None)})
    latents = sorted((OUTPUT_DIR / "h3_sample" / RUN_ID).glob("*.latent_summary.json"),
                     key=lambda p: p.stat().st_mtime) \
        if (OUTPUT_DIR / "h3_sample" / RUN_ID).is_dir() else []
    if latents:
        sample["latent_summary"] = json.loads(latents[-1].read_text(encoding="utf-8"))
        sample["latent_shapes_match"] = \
            sample["latent_summary"].get("shapes") == EXPECTED_LATENT_SHAPES
    PROBE["stages"]["sample"] = sample
    save()

print()
print("=" * 72)
print(f"status                 : {sample.get('status')} "
      f"({sample.get('failure_class') or ''}) in {sample.get('elapsed_s')} s")
print(f"sampler started after  : {guard_state.get('seconds_to_sampler_start')} s")
print(f"observed s/step        : {guard_state.get('observed_s_per_step')}")
print(f"projected total denoise: {guard_state.get('projected_denoise_total_s')} s")
print(f"peak ComfyUI RSS       : {sample.get('peak_comfy_rss_gib')} GiB")
print(f"min host available     : {sample.get('min_host_available_gib')} GiB")
print(f"peak VRAM / headroom   : {sample.get('peak_vram_used_mib')} / "
      f"{sample.get('min_vram_headroom_mib')} MiB")
print(f"latent shapes match    : {sample.get('latent_shapes_match')}")
if sample.get("perf_steps"):
    last = sample["perf_steps"][-1]
    print(f"perf probe (last step) : cuda {last.get('cuda_total_ms')} ms, "
          f"sdpa {last.get('sdpa_calls')} calls / {last.get('sdpa_ms')} ms, "
          f"alloc {last.get('alloc_delta')}")
    print(f"  per-class ms         : {last.get('per_class_ms')}")
if sample.get("turbo_lora_log"):
    print("-" * 72)
    for line in ([sample["turbo_lora_log"]] if isinstance(sample["turbo_lora_log"], str)
                 else sample["turbo_lora_log"]):
        print(f"  {line}")
    if sample.get("lora_keys_not_loaded"):
        print(f"  WARNING: {sample['lora_keys_not_loaded']} lora keys were not "
              "loaded; the LoRA is only partly applied.")
saved_latent = sample.get("saved_latent")
if saved_latent and "path" in saved_latent:
    print(f"latent saved           : {saved_latent['path']} "
          f"({saved_latent['bytes'] / 1024 ** 2:.1f} MiB)")
trace = sample.get("nan_trace")
if trace:
    culprit = trace.get("culprit")
    print("-" * 72)
    print(f"NaN trace: {trace.get('hooks')} modules hooked, "
          f"{trace.get('modules_seen')} produced a tensor"
          + (f", {trace['skipped']} unmeasurable" if trace.get("skipped") else ""))
    if culprit:
        out, inp = culprit.get("out", {}), culprit.get("in")
        print(f"  first non-finite : {culprit['module']}  ({culprit['cls']}, "
              f"{culprit['dtype']})")
        print(f"  its output       : nan={out.get('has_nan')} inf={out.get('has_inf')} "
              f"absmax={out.get('absmax')}")
        print(f"  its input        : "
              + (f"finite={inp.get('finite')} absmax={inp.get('absmax')}" if inp
                 else "no float tensor found"))
        print("  trail before it (max abs per module):")
        for rec in trace.get("trail", [])[-8:]:
            print(f"    {rec['order']:>6}  {str(rec['out'].get('absmax')):>12}  "
                  f"{rec['cls']:<20} {rec['module']}")
    elif trace.get("error"):
        print(f"  tracer error: {trace['error']}")
    elif trace.get("skipped"):
        print(f"  no culprit found, but {trace['skipped']} module(s) could not be "
              "measured -- the answer may be hiding in one of them:")
        for line in trace.get("errors", []):
            print(f"    {line}")
    else:
        print("  no module produced a non-finite output: the value is created "
              "outside the hooked modules (sampler arithmetic, or a functional call "
              "at the top level).")
print("=" * 72)
if sample.get("arch_error_seen"):
    print("The GGUF architecture patch did NOT take: 'Unexpected architecture type'.")
if sample.get("oom_seen"):
    print("A CUDA OOM appears in the log.")
if sample.get("status") not in ("SUCCESS", "CLIFF"):
    print(f"error: {sample.get('error') or sample.get('history_status')}")


## Stage 5 -- Verdict and benchmark ledger

One-look summary, plus an append-only ledger of every run on your Drive.


In [ ]:
# Stage 5 - verdict.
sample = PROBE["stages"].get("sample", {})
environment = PROBE["stages"]["environment"]
available_at_start = environment["host_ram_available_gib"]
peak = sample.get("peak_comfy_rss_gib")
dit_gib = round(DIT["bytes"] / GIB, 3)
# The encode stage on this runtime, for comparison.
ENCODE_REFERENCE_PEAK_GIB = 9.17

status = sample.get("status")
per_step = (sample.get("guard_state", {}).get("observed_s_per_step")
            or (sample.get("final_progress") or {}).get("observed_s_per_step"))
denoise_total_s = None
if status == "SUCCESS" and (sample.get("final_progress") or {}).get("elapsed_in_loop_s"):
    denoise_total_s = sample["final_progress"]["elapsed_in_loop_s"]
elif per_step:
    denoise_total_s = per_step * STEPS

verdict = {
    "dit_choice": DIT_CHOICE,
    "gear": GEAR,
    "gear_pixel_ratio": GEAR_PIXEL_RATIO,
    "dit_gib": dit_gib,
    "unet_dtype": UNET_DTYPE,
    "rmsnorm_fp32": RMSNORM_FP32,
    "nan_trace_culprit": (sample.get("nan_trace") or {}).get("culprit"),
    "fp32_text_branch": FP32_TEXT_BRANCH,
    "geometry": GEOMETRY,
    "steps": STEPS,
    "status": status,
    "failure_class": sample.get("failure_class"),
    "peak_comfy_rss_gib": peak,
    "host_available_at_start_gib": available_at_start,
    "headroom_gib": round(available_at_start - peak, 2) if peak is not None else None,
    "encode_stage_peak_gib": ENCODE_REFERENCE_PEAK_GIB,
    "peak_vs_encode_stage_gib": round(peak - ENCODE_REFERENCE_PEAK_GIB, 2)
    if peak is not None else None,
    "peak_vram_used_mib": sample.get("peak_vram_used_mib"),
    "min_vram_headroom_mib": sample.get("min_vram_headroom_mib"),
    "seconds_to_sampler_start": sample.get("guard_state", {}).get("seconds_to_sampler_start"),
    "observed_s_per_step": per_step,
    "denoise_total_s": round(denoise_total_s, 1) if denoise_total_s else None,
    "latent_shapes_match": sample.get("latent_shapes_match"),
    "architecture_patch_held": not sample.get("arch_error_seen", False),
    "sample_stage_confirmed": status == "SUCCESS" and sample.get("latent_shapes_match") is True,
}

if verdict["sample_stage_confirmed"]:
    minutes = (denoise_total_s or 0) / 60
    verdict["conclusion"] = (
        f"{DIT_CHOICE} ({dit_gib:.2f} GiB) sampled {STEPS} steps at "
        f"{GEOMETRY['width']}x{GEOMETRY['height']} / {GEOMETRY['length']} frames in "
        f"{minutes:.1f} min of denoise, peaking at {peak:.2f} GiB against "
        f"{available_at_start:.2f} GiB available, with no text encoder and no VAE in the "
        "graph. The sample stage of the three-stage split works on this runtime.")
elif status == "CLIFF" and sample.get("failure_class") == "SAMPLER_NEVER_STARTED" \
        and (peak or 0) < 3.0:
    # The DiT is 6-11 GiB; a peak near ComfyUI's baseline means it never loaded,
    # so nothing about the cliff was measured.
    verdict["sample_stage_confirmed"] = False
    verdict["conclusion"] = (
        f"The sampler never started AND peak RSS stayed at {peak} GiB, far below the "
        f"{dit_gib:.2f} GiB the DiT alone would occupy. The model never loaded, so this "
        "says nothing about the cliff. Look for a load error in the log rather than "
        "reaching for a smaller quant.")
elif status == "CLIFF":
    verdict["conclusion"] = (
        f"Past the cliff at this gear with {DIT_CHOICE}: {sample.get('failure_class')}. "
        + (f"Observed {per_step:.1f} s/step projects to "
           f"{(per_step * STEPS) / 60:.0f} min for {STEPS} steps. "
           if per_step else "")
        + "Try a smaller quant, or a smaller gear once conditioning exists for one. "
        "Note that the cliff is a precipice: the next step down may be fine.")
elif sample.get("arch_error_seen"):
    verdict["conclusion"] = (
        "ComfyUI-GGUF still rejected the architecture, so the IMG_ARCH_LIST patch did not "
        "reach the running process. Check Stage 1's recorded diff and that ComfyUI was "
        "restarted after patching.")
elif status == "FAILED" and sample.get("oom_seen"):
    verdict["conclusion"] = (
        "A CUDA OOM ended the run. Compare peak_vram_used_mib against the card and consider "
        "a smaller quant; host RAM was not the binding constraint here.")
else:
    # The history carries the node, the exception and the traceback. Reporting only
    # "FAILED" threw that away, which cost a round trip the first time it happened.
    failing = None
    for kind, data in (sample.get("history_status") or {}).get("messages", []):
        if kind == "execution_error":
            failing = data
    if failing:
        verdict["failing_node"] = {
            "node_id": failing.get("node_id"), "node_type": failing.get("node_type"),
            "exception_type": failing.get("exception_type"),
            "exception_message": (failing.get("exception_message") or "").strip()[:600],
            "executed_before_failure": failing.get("executed"),
            "traceback_tail": "".join(failing.get("traceback", [])[-2:])[-800:]}
        verdict["conclusion"] = (
            f"{failing.get('node_type')} (node {failing.get('node_id')}) raised "
            f"{failing.get('exception_type')}: "
            f"{(failing.get('exception_message') or '').strip()[:300]}")
    else:
        verdict["conclusion"] = f"Sampling did not complete: {sample.get('error') or status}."

PROBE["verdict"] = verdict
save()

print(json.dumps(verdict, indent=2, ensure_ascii=False))
print()
print("=" * 72)
print(f"DiT                    : {DIT_CHOICE}  {dit_gib:.2f} GiB")
print(f"gear                   : {GEAR}  ({GEAR_PIXEL_RATIO}x the pixels of gear 0.3, {GEOMETRY['width']}x{GEOMETRY['height']})")
print(f"status                 : {status} {sample.get('failure_class') or ''}")
print(f"peak RSS / available   : {peak} / {available_at_start} GiB "
      f"(encode stage peaked at {ENCODE_REFERENCE_PEAK_GIB})")
print(f"peak VRAM / headroom   : {verdict['peak_vram_used_mib']} / "
      f"{verdict['min_vram_headroom_mib']} MiB")
print(f"s/step                 : {per_step}")
print(f"denoise total          : {verdict['denoise_total_s']} s")
print(f"latent shapes match    : {verdict['latent_shapes_match']}")
print(f"architecture patch held: {verdict['architecture_patch_held']}")
print("-" * 72)
print(f"SAMPLE STAGE CONFIRMED : {verdict['sample_stage_confirmed']}")
print("=" * 72)
print(verdict["conclusion"])
if verdict.get("failing_node"):
    print()
    print("traceback tail:")
    print(verdict["failing_node"]["traceback_tail"])
print()
print("Not measured here: the decode stage (VAEs), and the cliff position across gears --")
print("that needs a conditioning blob per gear, which the encode stage produces.")
print(f"Full result: {RESULT_PATH}")

# --- append this run to the campaign ledger ---------------------------------
ledger_row = {
    "measured_at": utc_now(), "run_id": RUN_ID, "gear": GEAR,
    "geometry": GEOMETRY, "gear_pixel_ratio": GEAR_PIXEL_RATIO,
    "clip_seconds": CLIP_SECONDS, "dit_choice": DIT_CHOICE,
    "unet_dtype": UNET_DTYPE, "turbo_lora": TURBO_LORA,
    "lora_strength": LORA_STRENGTH if LORA else None,
    "lora_low_vram": LORA_LOW_VRAM if LORA else None, "steps": STEPS,
    "status": verdict.get("status"), "failure_class": verdict.get("failure_class"),
    "observed_s_per_step": verdict.get("observed_s_per_step"),
    "denoise_total_s": verdict.get("denoise_total_s"),
    "seconds_to_sampler_start": verdict.get("seconds_to_sampler_start"),
    "peak_comfy_rss_gib": verdict.get("peak_comfy_rss_gib"),
    "peak_vram_used_mib": verdict.get("peak_vram_used_mib"),
    "min_vram_headroom_mib": verdict.get("min_vram_headroom_mib"),
    "latent_shapes_match": verdict.get("latent_shapes_match"),
    "sample_stage_confirmed": verdict.get("sample_stage_confirmed"),
}
ledger_written = None
if DRIVE_MOUNTED:
    try:
        CLIFF_LEDGER.parent.mkdir(parents=True, exist_ok=True)
        with CLIFF_LEDGER.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(ledger_row, ensure_ascii=False) + "\n")
        ledger_written = str(CLIFF_LEDGER)
    except Exception as exc:
        print(f"ledger append failed: {type(exc).__name__}: {exc}")
verdict["ledger"] = ledger_written
PROBE["verdict"] = verdict
save()

# --- the campaign so far -----------------------------------------------------
rows = []
if DRIVE_MOUNTED and CLIFF_LEDGER.is_file():
    for line in CLIFF_LEDGER.read_text(encoding="utf-8").splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except ValueError:
                pass
# One row per (gear, steps, dtype, lora): a re-run supersedes its predecessor.
latest = {}
for row in rows:
    latest[(row.get("gear"), row.get("steps"), row.get("unet_dtype"),
            row.get("turbo_lora"))] = row
campaign = sorted(latest.values(), key=lambda r: float(r.get("gear") or 0))
if len(campaign) > 1:
    print()
    print("=" * 72)
    print("cliff campaign so far (from the Drive ledger)")
    print(f"{'gear':<6}{'geometry':<16}{'px':>7}{'s/step':>9}{'x0.3':>7}"
          f"{'total':>9}{'VRAM':>8}  status")
    # gear 0.3 at these settings was measured before this ledger existed. Use it
    # as the baseline when the ledger has no 0.3 row of its own, and say so --
    # a ratio against a number from a different day should be labelled as one.
    BASELINE_0_3_S_PER_STEP = 235.0   # bf16, Q4_K_M, v4-600-EMA, 6 step, 2026-08-14
    base = next((r["observed_s_per_step"] for r in campaign
                 if r.get("gear") == "0.3" and r.get("observed_s_per_step")), None)
    base_source = "this ledger"
    if base is None:
        base, base_source = BASELINE_0_3_S_PER_STEP, "the 2026-08-14 run, not this ledger"
    for row in campaign:
        geometry = row.get("geometry") or {}
        shape = "{}x{}".format(geometry.get("width"), geometry.get("height"))
        per_step = row.get("observed_s_per_step")
        ratio = f"{per_step / base:.2f}" if (per_step and base) else "-"
        total = row.get("denoise_total_s")
        print(f"{row.get('gear', ''):<6}{shape:<16}"
              f"{row.get('gear_pixel_ratio', ''):>7}"
              f"{(round(per_step, 1) if per_step else '-'):>9}{ratio:>7}"
              f"{(str(round(total / 60, 1)) + 'm' if total else '-'):>9}"
              f"{str(row.get('peak_vram_used_mib')):>8}  "
              f"{row.get('status')}"
              + (f" / {row['failure_class']}" if row.get("failure_class") else ""))
    print()
    print(f"x0.3 is the measured time ratio against gear 0.3 ({base:.0f} s/step, from "
          f"{base_source}).")
    print("The pixel ratio is a")
    print("lower bound: attention is quadratic in tokens, so a cliff shows up as")
    print("x0.3 running far ahead of px.")
    print("=" * 72)

stop_comfy()
print("ComfyUI stopped.")


## Stage 6 -- Decode to MP4 (in a fresh process -- that IS the trick)

ComfyUI is restarted so the DiT is gone before the VAEs load; peak stays at
max(stage) instead of sum. Audio is wired explicitly (it is an optional input --
forget it and you silently get a silent MP4). Black-frame QA included.


In [ ]:
# Stage 6 - decode the sampled latent into an MP4.
#
# This restarts ComfyUI first. The DiT is 10.77 GiB and the video VAE is 4.85 GiB
# against a 12.67 GiB host, so decoding inside the process that just sampled would
# ask for the sum of two stages. The whole three-stage split exists to keep the
# peak at max(stage), and that only holds if the stages really are separate
# processes. The latent crossed the boundary as a file.
#
# It also starts without the sampling dtype flag: --force-fp16 was a statement
# about the DiT, and the audio VAE ships as fp32.
def decode_fill(node_type, known, schema):
    """Supply every required input of node_type, from `known` or from the schema.

    Node signatures drift between ComfyUI versions. Rather than hard-coding the
    widget order seen in a workflow JSON, take the running server's schema as the
    truth, and fail loudly with the offending entry printed when something
    required has no value here.
    """
    spec_in = schema[node_type].get("input", {})
    required = spec_in.get("required") or {}
    optional = spec_in.get("optional") or {}
    inputs, unresolved = {}, []
    # Anything named here that the node accepts goes in, required or not. CreateVideo
    # lists `audio` as optional, and filling only the required inputs would produce
    # a silent MP4 that looks like a success.
    for name, value in known.items():
        if name in optional and name not in required:
            inputs[name] = value
    unknown = sorted(set(known) - set(required) - set(optional))
    if unknown:
        raise RuntimeError(f"{node_type} does not accept {unknown}; its inputs are "
                           f"{sorted(set(required) | set(optional))}. The node "
                           "signature changed and this graph would be wired wrong.")
    for name, spec in required.items():
        if name in known:
            inputs[name] = known[name]
            continue
        options = spec[0] if spec else None
        config = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
        dynamic = config.get("options") if options == "COMFY_DYNAMICCOMBO_V3" else None
        if dynamic:
            # A dynamic combo does not put its choices in spec[0]. They live in
            # config["options"] as {"key": ..., "inputs": {...}}, and the value the
            # API wants is the bare key -- ComfyUI expands the chosen option's own
            # inputs itself (comfy_api/latest/_io.py: key = live_inputs[id]).
            # Prefer a choice that needs nothing further; "auto" is that choice for
            # SaveVideo's codec.
            def needs_nothing(option):
                return not ((option.get("inputs") or {}).get("required") or {})
            choice = next((o for o in dynamic if o.get("key") == "auto" and needs_nothing(o)),
                          next((o for o in dynamic if needs_nothing(o)), None))
            if choice is None:
                unresolved.append(
                    f"{name}: every option of this dynamic combo needs further "
                    f"inputs ({[o.get('key') for o in dynamic]})")
            else:
                inputs[name] = choice["key"]
        elif isinstance(options, list) and options:
            inputs[name] = config.get("default", options[0])
        elif "default" in config:
            inputs[name] = config["default"]
        else:
            unresolved.append(f"{name}: {spec}")
    if unresolved:
        raise RuntimeError(f"{node_type} requires inputs this stage has no value "
                           "for: " + "; ".join(unresolved))
    return inputs


video = {"requested": DECODE_TO_MP4, "fps": FPS}
sample = PROBE["stages"].get("sample", {})
latent_path = LATENT_STORE / f"{LATENT_KEY}.pt" if "LATENT_KEY" in globals() else None
latent_finite = (sample.get("latent_summary") or {}).get("finite")

if not DECODE_TO_MP4:
    video["status"] = "SKIPPED_OFF"
elif latent_path is None or not latent_path.is_file():
    video["status"] = "SKIPPED_NO_LATENT"
    video["looked_for"] = str(latent_path)
elif latent_finite is not None and not all(latent_finite):
    # Decoding a non-finite latent produces a black or noise video and says
    # nothing about whether decoding works. Refuse, and say which part is bad.
    video["status"] = "SKIPPED_NON_FINITE_LATENT"
    video["latent_finite"] = latent_finite
else:
    video["latent_path"] = str(latent_path)
    video["latent_mib"] = round(latent_path.stat().st_size / 1024 ** 2, 1)
    started_decode = time.perf_counter()
    stop_comfy()
    decode_stats, decode_log_offset = start_comfy(dtype_flag=None)
    decode_object_info = requests.get(f"{BASE_URL}/object_info", timeout=300).json()
    video["argv"] = decode_stats.get("system", {}).get("argv")

    video_prefix = f"h3_video/{RUN_ID}/{label}"
    decode_graph = {
        "1": {"class_type": "VAELoader", "inputs": decode_fill("VAELoader", {
            "vae_name": Path(VAE_FILES["video"]["file"]).name}, decode_object_info)},
        "2": {"class_type": "VAELoader", "inputs": decode_fill("VAELoader", {
            "vae_name": Path(VAE_FILES["audio"]["file"]).name}, decode_object_info)},
        "3": {"class_type": "H3LoadLatent", "inputs": decode_fill("H3LoadLatent", {
            "store_dir": str(LATENT_STORE), "key": LATENT_KEY}, decode_object_info)},
        # Both decoders take the same AV latent and each picks its own half.
        "4": {"class_type": "VAEDecode", "inputs": decode_fill("VAEDecode", {
            "samples": ["3", 0], "vae": ["1", 0]}, decode_object_info)},
        "5": {"class_type": "VAEDecodeAudio", "inputs": decode_fill("VAEDecodeAudio", {
            "samples": ["3", 0], "vae": ["2", 0]}, decode_object_info)},
        "6": {"class_type": "CreateVideo", "inputs": decode_fill("CreateVideo", {
            "images": ["4", 0], "audio": ["5", 0], "fps": float(FPS)}, decode_object_info)},
        "7": {"class_type": "SaveVideo", "inputs": decode_fill("SaveVideo", {
            "video": ["6", 0], "filename_prefix": video_prefix}, decode_object_info)},
    }
    (WORK / "decode_graph.json").write_text(json.dumps(decode_graph, indent=2),
                                            encoding="utf-8")
    video["graph"] = decode_graph
    print("Decode graph: " + ", ".join(
        f"{i}:{s['class_type']}" for i, s in sorted(decode_graph.items())))
    print("  no UnetLoaderGGUF in it. Resident weights: the two VAEs "
          f"({sum(s['bytes'] for s in VAE_FILES.values()) / GIB:.2f} GiB)")

    decode_samples, decode_stop = [], threading.Event()

    def watch_decode():
        while not decode_stop.is_set():
            record = {"host_available_gib": round(psutil.virtual_memory().available / GIB, 3)}
            try:
                root = psutil.Process(COMFY_PROCESS.pid)
                rss = root.memory_info().rss
                for child in root.children(recursive=True):
                    try:
                        rss += child.memory_info().rss
                    except psutil.Error:
                        pass
                record["comfy_rss_gib"] = round(rss / GIB, 3)
            except Exception:
                pass
            try:
                line = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used,memory.total",
                     "--format=csv,noheader,nounits"], text=True, timeout=5,
                    env=scrubbed_child_environment()).strip().splitlines()[0]
                used, total = [float(x) for x in line.split(",")]
                record["vram_used_mib"], record["vram_headroom_mib"] = used, total - used
            except Exception:
                pass
            decode_samples.append(record)
            decode_stop.wait(0.5)

    decode_watcher = threading.Thread(target=watch_decode, daemon=True)
    decode_watcher.start()
    decode_prompt_id = str(uuid.uuid4())
    decode_record = None
    try:
        response = requests.post(f"{BASE_URL}/prompt", timeout=120, json={
            "prompt": decode_graph, "client_id": str(uuid.uuid4()),
            "prompt_id": decode_prompt_id})
        response.raise_for_status()
        queued = response.json()
        if queued.get("node_errors"):
            raise RuntimeError(f"Node errors: {queued['node_errors']}")
        print("submitted; decoding")
        shown = decode_log_offset
        deadline = time.perf_counter() + DECODE_TIMEOUT_S
        while time.perf_counter() < deadline:
            if COMFY_PROCESS.poll() is not None:
                raise RuntimeError("ComfyUI exited during decoding (check for an OOM kill)")
            with COMFY_LOG.open("rb") as reader:
                reader.seek(shown)
                chunk = reader.read().decode("utf-8", errors="replace")
                shown += len(chunk.encode("utf-8"))
            for line in chunk.splitlines():
                if line.strip():
                    print("    " + line[:160])
            try:
                history = requests.get(f"{BASE_URL}/history/{decode_prompt_id}",
                                       timeout=60).json()
                if decode_prompt_id in history:
                    status = history[decode_prompt_id].get("status", {})
                    if status.get("completed") is True or status.get("status_str") == "error":
                        decode_record = history[decode_prompt_id]
                        break
            except Exception:
                pass
            time.sleep(3)
        else:
            raise TimeoutError(f"Decoding did not finish within {DECODE_TIMEOUT_S}s")
        status = decode_record.get("status", {})
        video["status_str"] = status.get("status_str")
        video["status"] = "SUCCESS" if status.get("status_str") == "success" else "FAILED"
        if video["status"] != "SUCCESS":
            video["history_status"] = status
    except Exception as exc:
        video["status"] = "FAILED"
        video["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        decode_stop.set()
        decode_watcher.join(timeout=5)
        video["elapsed_s"] = round(time.perf_counter() - started_decode, 1)
        decode_log = tail_comfy_log(decode_log_offset)
        (WORK / f"decode_log_{label}.txt").write_text(decode_log, encoding="utf-8")
        video["oom_seen"] = bool(re.search(r"CUDA out of memory|torch\.OutOfMemoryError",
                                           decode_log, re.I))
        if decode_samples:
            rss = [s["comfy_rss_gib"] for s in decode_samples if "comfy_rss_gib" in s]
            video.update({
                "peak_comfy_rss_gib": max(rss) if rss else None,
                "min_host_available_gib": min(s["host_available_gib"] for s in decode_samples),
                "peak_vram_used_mib": max((s["vram_used_mib"] for s in decode_samples
                                           if "vram_used_mib" in s), default=None),
                "min_vram_headroom_mib": min((s["vram_headroom_mib"] for s in decode_samples
                                              if "vram_headroom_mib" in s), default=None)})
        if decode_record is not None:
            (WORK / f"decode_history_{label}.json").write_text(
                json.dumps(decode_record, indent=2), encoding="utf-8")

    # --- find what SaveVideo wrote -------------------------------------------
    produced = []
    for node_output in ((decode_record or {}).get("outputs") or {}).values():
        for entry in node_output.get("images", []) + node_output.get("videos", []) \
                + node_output.get("gifs", []):
            if isinstance(entry, dict) and entry.get("filename"):
                produced.append(OUTPUT_DIR / entry.get("subfolder", "") / entry["filename"])
    if not produced:
        # History did not name it; fall back to the prefix we asked for.
        directory = OUTPUT_DIR / "h3_video" / RUN_ID
        produced = sorted(directory.glob("*"), key=lambda p: p.stat().st_mtime) \
            if directory.is_dir() else []
    produced = [p for p in produced if p.is_file()]
    video["produced"] = [str(p) for p in produced]
    if produced:
        mp4 = produced[-1]
        video["file"] = str(mp4)
        video["bytes"] = mp4.stat().st_size
        video["mib"] = round(mp4.stat().st_size / 1024 ** 2, 2)

        # --- is there actually a picture in it? ------------------------------
        # A latent can be finite and still decode to black, and "the file exists"
        # is not the same claim as "the video has content".
        inspection = {}
        try:
            import av

            with av.open(str(mp4)) as container:
                streams = {s.type: s for s in container.streams}
                inspection["streams"] = sorted(streams)
                vstream = streams.get("video")
                if vstream is not None:
                    inspection["video"] = {
                        "codec": vstream.codec_context.name,
                        "width": vstream.codec_context.width,
                        "height": vstream.codec_context.height,
                        "frames": vstream.frames,
                        "duration_s": round(float(vstream.duration * vstream.time_base), 3)
                        if vstream.duration else None,
                        "avg_rate": float(vstream.average_rate) if vstream.average_rate else None,
                    }
                astream = streams.get("audio")
                if astream is not None:
                    inspection["audio"] = {
                        "codec": astream.codec_context.name,
                        "sample_rate": astream.codec_context.sample_rate,
                        "channels": astream.codec_context.channels,
                        "duration_s": round(float(astream.duration * astream.time_base), 3)
                        if astream.duration else None,
                    }
            with av.open(str(mp4)) as container:
                brightness, index = [], 0
                for frame in container.decode(video=0):
                    if index % VIDEO_INSPECT_EVERY == 0:
                        plane = frame.to_ndarray(format="gray")
                        brightness.append((index, float(plane.mean()), float(plane.std())))
                    index += 1
                inspection["frames_decoded"] = index
                inspection["sampled"] = [
                    {"frame": i, "mean": round(m, 2), "std": round(s, 2)}
                    for i, m, s in brightness]
                if brightness:
                    means = [m for _, m, _ in brightness]
                    stds = [s for _, _, s in brightness]
                    inspection["mean_luma"] = round(sum(means) / len(means), 2)
                    inspection["mean_std"] = round(sum(stds) / len(stds), 2)
                    # Near-black: dark AND flat. A dark but textured night shot is
                    # a picture; a dark and flat frame is a failure.
                    inspection["near_black_frames"] = sum(
                        1 for m, s in zip(means, stds) if m < 4.0 and s < 2.0)
                    inspection["frames_sampled"] = len(means)
                    inspection["all_frames_identical"] = (
                        len(set(round(m, 3) for m in means)) == 1 and len(means) > 1)
        except Exception as exc:
            inspection["error"] = f"{type(exc).__name__}: {exc}"
        video["inspection"] = inspection

        # --- keep it: the runtime is ephemeral -----------------------------
        if DRIVE_MOUNT.is_dir() and (DRIVE_MOUNT / "MyDrive").is_dir():
            try:
                DRIVE_VIDEO_STORE.mkdir(parents=True, exist_ok=True)
                copied = DRIVE_VIDEO_STORE / f"{RUN_ID}_{label}{mp4.suffix}"
                shutil.copy2(mp4, copied)
                video["drive_copy"] = str(copied)
            except Exception as exc:
                video["drive_copy_error"] = f"{type(exc).__name__}: {exc}"

PROBE["stages"]["video"] = video
PROBE.setdefault("verdict", {})["video"] = {
    "status": video.get("status"),
    "file": video.get("file"),
    "drive_copy": video.get("drive_copy"),
    "elapsed_s": video.get("elapsed_s"),
    "peak_vram_used_mib": video.get("peak_vram_used_mib"),
    "peak_comfy_rss_gib": video.get("peak_comfy_rss_gib"),
    "inspection": video.get("inspection"),
}
save()

print()
print("=" * 72)
print(f"decode status          : {video.get('status')}")
if video.get("status") == "SUCCESS":
    print(f"file                   : {video.get('file')}  ({video.get('mib')} MiB)")
    if video.get("drive_copy"):
        print(f"copied to Drive        : {video['drive_copy']}")
    print(f"decode elapsed         : {video.get('elapsed_s')} s "
          "(includes restarting ComfyUI and loading the VAEs)")
    print(f"peak VRAM / headroom   : {video.get('peak_vram_used_mib')} / "
          f"{video.get('min_vram_headroom_mib')} MiB")
    print(f"peak RSS / available   : {video.get('peak_comfy_rss_gib')} / "
          f"{video.get('min_host_available_gib')} GiB")
    inspection = video.get("inspection") or {}
    if inspection.get("error"):
        print(f"inspection failed      : {inspection['error']}")
    else:
        print(f"streams                : {inspection.get('streams')}")
        if inspection.get("video"):
            v = inspection["video"]
            print(f"  video                : {v.get('width')}x{v.get('height')} "
                  f"{v.get('codec')}  {inspection.get('frames_decoded')} frames  "
                  f"{v.get('duration_s')} s  @{v.get('avg_rate')} fps")
        if inspection.get("audio"):
            a = inspection["audio"]
            print(f"  audio                : {a.get('codec')} {a.get('sample_rate')} Hz "
                  f"{a.get('channels')} ch  {a.get('duration_s')} s")
        print(f"  luma mean / texture  : {inspection.get('mean_luma')} / "
              f"{inspection.get('mean_std')}  "
              f"(near-black {inspection.get('near_black_frames')}"
              f"/{inspection.get('frames_sampled')} sampled)")
        if inspection.get("near_black_frames") == inspection.get("frames_sampled"):
            print("  EVERY sampled frame is dark and flat. The file exists but "
                  "there is no picture in it.")
        elif inspection.get("all_frames_identical"):
            print("  every sampled frame has the same brightness: possibly a still.")
elif video.get("status") == "SKIPPED_NON_FINITE_LATENT":
    print("The sampled latent is not finite, so there is nothing worth decoding. "
          f"finite = {video.get('latent_finite')}")
elif video.get("status") == "SKIPPED_NO_LATENT":
    print(f"No latent at {video.get('looked_for')}. Run Stage 4 first.")
elif video.get("status") == "SKIPPED_OFF":
    print("DECODE_TO_MP4 is False.")
else:
    print(f"error: {video.get('error') or video.get('history_status')}")
    if video.get("oom_seen"):
        print("A CUDA OOM appears in the decode log.")
print("=" * 72)

# --- inline playback ----------------------------------------------------------
# Embed the MP4 as a base64 data URL so it plays right here in the notebook.
# Size-capped: a 5-second clip is well under a megabyte, but a long experiment
# should not balloon the saved .ipynb.
if video.get("status") == "SUCCESS" and video.get("file"):
    _mp4_path = Path(video["file"])
    _INLINE_CAP_MIB = 25
    if not _mp4_path.is_file():
        print(f"inline playback: file not found at {_mp4_path}")
    elif _mp4_path.stat().st_size > _INLINE_CAP_MIB * 2**20:
        print(f"inline playback: skipped "
              f"({_mp4_path.stat().st_size / 2**20:.1f} MiB > {_INLINE_CAP_MIB} MiB "
              "cap); play it from Drive instead")
    else:
        from base64 import b64encode
        from IPython.display import HTML, display
        _data_url = ("data:video/mp4;base64,"
                     + b64encode(_mp4_path.read_bytes()).decode())
        display(HTML(
            f'<video width="{GEOMETRY["width"]}" controls loop '
            'style="max-width:100%">'
            f'<source src="{_data_url}" type="video/mp4"></video>'))


## Stage 7 -- Video Upscaler 1080p / 4K (Super-Resolução Neural + Áudio Sincronizado)

Esta célula realiza o **upscaling com super-resolução neural** do vídeo gerado no Stage 6:
* **Modelos Disponíveis:** `4x-UltraSharp` (recomendado para máximo detalhe e nitidez) ou `RealESRGAN_x4plus`.
* **Áudio Preservado e Sincronizado:** Extrai e remonta a trilha de áudio original do MiniMax H3 via FFmpeg mantendo sincronia labial e sonora perfeita.
* **Pipeline Eficiente na GPU:** Processamento frame a frame na Tesla T4 sem gravar imagens temporárias no disco, com codificação direta via pipe H.264.
* **Resoluções Suportadas:** **1080p (Full HD)**, **2x** ou **4x**.
* **Armazenamento Permanente:** O vídeo upscaled é salvo diretamente no seu Google Drive em `MyDrive/H3/MiniMax-H3-Native-Colab/video_upscaled/`.


In [ ]:
# =============================================================================
# Stage 7 - Video Upscaler 1080p / 4K (Super-Resolução + Áudio Sincronizado)
# =============================================================================
# @title 🎬 Upscale de Vídeo com Áudio Preservado { run: "auto" }
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path
import cv2
import torch
from tqdm.auto import tqdm

# --- Parâmetros de Configuração ---
UPSCALE_MODEL_CHOICE = "4x-UltraSharp"  # @param ["4x-UltraSharp", "RealESRGAN_x4plus"]
UPSCALE_RESOLUTION = "1080p (Full HD)"  # @param ["1080p (Full HD)", "2x", "4x"]
CRF_QUALITY = 17  # @param {type:"slider", min:14, max:24, step:1}
CUSTOM_VIDEO_PATH = ""  # @param {type:"string"}

# 1. Localizar o vídeo de entrada
target_video = None
if CUSTOM_VIDEO_PATH.strip() and Path(CUSTOM_VIDEO_PATH.strip()).is_file():
    target_video = Path(CUSTOM_VIDEO_PATH.strip())
elif "video" in globals() and isinstance(globals()["video"], dict) and globals()["video"].get("file"):
    f = Path(globals()["video"]["file"])
    if f.is_file():
        target_video = f

if not target_video:
    # Busca o vídeo mais recente na pasta de saída do Drive ou local
    candidates = []
    if "DRIVE_VIDEO_STORE" in globals() and globals()["DRIVE_VIDEO_STORE"].is_dir():
        candidates.extend(globals()["DRIVE_VIDEO_STORE"].glob("*.mp4"))
    if "OUTPUT_DIR" in globals() and globals()["OUTPUT_DIR"].is_dir():
        candidates.extend(globals()["OUTPUT_DIR"].glob("*.mp4"))
    if candidates:
        target_video = max(candidates, key=lambda p: p.stat().st_mtime)

if not target_video or not target_video.is_file():
    raise RuntimeError("Nenhum vídeo MP4 encontrado para upscale. Execute o Stage 6 primeiro ou forneça um caminho em CUSTOM_VIDEO_PATH.")

print(f"🎬 Vídeo de entrada selecionado: {target_video.name}")
print(f"   Tamanho: {target_video.stat().st_size / (1024**2):.2f} MiB | Caminho: {target_video}")

# 2. Configurar pastas de destino
if "DRIVE_H3" in globals():
    UPSCALED_DIR = globals()["DRIVE_H3"] / "video_upscaled"
elif "DRIVE_VIDEO_STORE" in globals():
    UPSCALED_DIR = globals()["DRIVE_VIDEO_STORE"].parent / "video_upscaled"
else:
    UPSCALED_DIR = Path("/content/drive/MyDrive/H3/MiniMax-H3-Native-Colab/video_upscaled")
UPSCALED_DIR.mkdir(parents=True, exist_ok=True)

if "DRIVE_COMFY_MODELS" in globals():
    UPSCALE_MODELS_DIR = globals()["DRIVE_COMFY_MODELS"] / "upscale_models"
elif "COMFY_DIR" in globals():
    UPSCALE_MODELS_DIR = globals()["COMFY_DIR"] / "models" / "upscale_models"
else:
    UPSCALE_MODELS_DIR = Path("/content/drive/MyDrive/ComfyUI/models/upscale_models")
UPSCALE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 3. Garantir instalação do spandrel para carregamento de modelos
try:
    from spandrel import ModelLoader
except ImportError:
    print("📦 Instalando biblioteca de super-resolução (spandrel)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spandrel"], check=True)
    from spandrel import ModelLoader

# 4. Catálogo e verificação inteligente do modelo de upscale
UPSCALE_CATALOG = {
    "4x-UltraSharp": {
        "file": "4x-UltraSharp.pth",
        "repo": "lokcx/4x-Ultrasharp",
        "bytes": 66986421,
    },
    "RealESRGAN_x4plus": {
        "file": "RealESRGAN_x4plus.pth",
        "repo": "ai-forever/Real-ESRGAN",
        "bytes": 67021162,
    }
}

spec = UPSCALE_CATALOG[UPSCALE_MODEL_CHOICE]
model_dest = UPSCALE_MODELS_DIR / spec["file"]

# Procura se o modelo já existe no Drive ou local
found_model = None
if "find_model_file" in globals():
    found_model = find_model_file(spec["file"], subfolders=["upscale_models"], min_bytes=spec["bytes"])
if not found_model and model_dest.is_file():
    found_model = model_dest

if found_model and found_model.is_file():
    print(f"✅ Reutilizando modelo de upscale do Drive: {found_model.name}")
    model_path = found_model
else:
    print(f"⬇️ Baixando modelo {spec['file']} direto para {UPSCALE_MODELS_DIR.name}...")
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(repo_id=spec["repo"], filename=spec["file"], local_dir=str(UPSCALE_MODELS_DIR))
    model_path = Path(downloaded)
    if model_path != model_dest and model_path.is_file():
        shutil.move(str(model_path), str(model_dest))
        model_path = model_dest
    print("✅ Download do modelo de super-resolução concluído!")

# 5. Carregar modelo na GPU (Tesla T4)
print(f"🧠 Carregando {model_path.name} na GPU...")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
loader = ModelLoader()
upscale_model = loader.load_from_file(str(model_path)).eval().to(device)
use_fp16 = torch.cuda.is_available()
if use_fp16:
    upscale_model = upscale_model.half()

# 6. Analisar vídeo de entrada com OpenCV
cap = cv2.VideoCapture(str(target_video))
if not cap.isOpened():
    raise RuntimeError(f"Falha ao abrir vídeo {target_video}")

in_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
in_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Calcular resolução final
if UPSCALE_RESOLUTION == "2x":
    out_w, out_h = in_w * 2, in_h * 2
elif UPSCALE_RESOLUTION == "4x":
    out_w, out_h = in_w * 4, in_h * 4
else:  # 1080p
    out_h = 1080
    out_w = int(round((in_w * (1080 / in_h)) / 2) * 2)  # Garante largura par divisível por 2

print(f"📊 Resolução original: {in_w}x{in_h} @ {fps:.1f} fps ({total_frames} frames)")
print(f"🚀 Resolução final após Upscale: {out_w}x{out_h} ({UPSCALE_RESOLUTION})")

# 7. Extrair trilha de áudio original para manter sincronização perfeita
temp_audio = Path("/content/temp_h3_audio.aac")
has_audio = False
try:
    audio_probe = subprocess.run([
        "ffmpeg", "-y", "-i", str(target_video), "-vn", "-c:a", "copy", str(temp_audio)
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if temp_audio.is_file() and temp_audio.stat().st_size > 1000:
        has_audio = True
        print(f"🎵 Trilha de áudio extraída com sucesso ({temp_audio.stat().st_size / 1024:.1f} KB)")
except Exception as e:
    print(f"⚠️ Aviso na extração de áudio: {e}")

# 8. Pipeline FFmpeg de codificação via stdin (Zero frames temporários em disco!)
output_filename = f"{target_video.stem}_upscaled_{out_w}x{out_h}.mp4"
output_path = UPSCALED_DIR / output_filename

ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-f", "rawvideo",
    "-vcodec", "rawvideo",
    "-s", f"{out_w}x{out_h}",
    "-pix_fmt", "bgr24",
    "-r", str(fps),
    "-i", "-"
]

if has_audio:
    ffmpeg_cmd.extend(["-i", str(temp_audio), "-c:a", "copy"])

ffmpeg_cmd.extend([
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-crf", str(CRF_QUALITY),
    "-preset", "fast",
    "-movflags", "+faststart",
    str(output_path)
])

ffmpeg_proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)

# 9. Loop de processamento de super-resolução na GPU
print(f"⚡ Processando {total_frames} frames na GPU...")
t_start = time.perf_counter()

try:
    with tqdm(total=total_frames, desc="Upscaling", unit="frame") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                break
            
            # Converte BGR (H, W, C) para Tensor (1, C, H, W) normalizado
            t_frame = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).float() / 255.0
            t_frame = t_frame.to(device=device)
            if use_fp16:
                t_frame = t_frame.half()
                
            with torch.no_grad():
                sr_tensor = upscale_model(t_frame)
                
            # Retorna para numpy uint8
            sr_frame = (sr_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1) * 255.0).byte().cpu().numpy()
            
            # Redimensionamento final caso a resolução pretendida seja diferente do fator 4x nativo
            if (sr_frame.shape[1], sr_frame.shape[0]) != (out_w, out_h):
                sr_frame = cv2.resize(sr_frame, (out_w, out_h), interpolation=cv2.INTER_AREA if sr_frame.shape[1] > out_w else cv2.INTER_LANCZOS4)
                
            ffmpeg_proc.stdin.write(sr_frame.tobytes())
            pbar.update(1)
            
except Exception as exc:
    print(f"❌ Erro durante o processamento: {exc}")
    ffmpeg_proc.kill()
    raise
finally:
    cap.release()
    if ffmpeg_proc.stdin:
        ffmpeg_proc.stdin.close()
    ffmpeg_proc.wait()
    if temp_audio.is_file():
        temp_audio.unlink(missing_ok=True)

duration = time.perf_counter() - t_start
fps_proc = total_frames / max(duration, 0.001)

print("\n" + "=" * 72)
print("🎉 VÍDEO UPSCALED COM SUCESSO!")
print(f"📁 Arquivo Salvo : {output_path}")
print(f"📊 Resolução     : {out_w}x{out_h}")
print(f"⏱️ Tempo Total   : {duration:.1f}s ({fps_proc:.1f} fps na GPU)")
print(f"💾 Tamanho       : {output_path.stat().st_size / (1024**2):.2f} MiB")
print("=" * 72)

# 10. Player de vídeo inline no Colab
try:
    from base64 import b64encode
    from IPython.display import HTML, display
    _data_url = "data:video/mp4;base64," + b64encode(output_path.read_bytes()).decode()
    display(HTML(f'''
    <div style="text-align: center; margin-top: 15px;">
        <h4 style="margin-bottom: 8px;">🎬 Pré-visualização do Vídeo em Alta Resolução ({out_w}x{out_h})</h4>
        <video width="{min(out_w, 960)}" controls autoplay loop style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3); max-width: 100%;">
            <source src="{_data_url}" type="video/mp4">
        </video>
    </div>
    '''))
except Exception:
    pass
